# learner

> The learner and the shelf: the settings, the prompts, the store, what you know, how you learn, the jobs and the stored walks.

This module is the base of the engine. Every other module imports from it, and it imports from no other module of abhyasa.
It holds what abhyasa knows about the learner, and the shelf where it keeps it. The parts come in the order that they import each other:

- **Settings**: the settings file, the settings table and its defaults.
- **Prompts**: the system prompts that you can see and change in Settings.
- **The store**: the learning shelf in your vault. Each concept, attempt and session is one document.
- **The learner model**: the depth, the floor and the ceiling of each concept, and when to review it.
- **Ways to learn**: which way of teaching helps you, such as code or an analogy.
- **The profile**: your background, goals, interests and pace.
- **Background jobs**: the stored rows of the jobs that the activity chip shows.
- **Stored walks**: read and save a walk, find its current idea, and count its steps.

Two functions of the profile ask a model. They import the model layer (`abhyasa.models`) when they run, because that layer imports this module.

In [ ]:
#| default_exp learner

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

In [ ]:
#| hide
from fastcore.test import *

Import the modules that this module uses. The heavy ones load inside the functions that need them, so the import stays light.

In [ ]:
#| export
import threading, weakref, os, json, time, random, uuid, contextvars
from contextlib import contextmanager
from fastcore.utils import *
from pathlib import Path

## Settings

One table for each value that you can change, and the file that keeps your changes.

Each value that a person can change is a row in one table, `SETTINGS`. A row has a group, a name, a label, one line of help, a type, a default and, for a number, a range.
The engine, the web app and the Settings page all read this table. Thus each value has one place.

Your changes go in `settings.json` in `cfg_dir()`. The code reads a value when it uses it, so a change works at once, with no restart.
A saved value outside its range gets the nearest end of the range. A saved value of the wrong type gets the default. A bad file never stops the app.

This module imports only the standard library and fastcore. Each engine module reads its values from here, so the import must stay light.

### The groups

The Settings page shows the groups in this order. Each group has a label and one lead line.

In [ ]:
#| export
GROUPS = dict(look=('Look', 'Pick the colours of the app.'),
              learning=('Learning', 'Set how the tutor teaches you. The Profile page shows the same values.'),
              walks=('Walks', 'Pick how long a walk is and how much of it is hands-on.'),
              talks=('Talks', 'Set how a talk runs and how the room works.'),
              checks=('Checks and grading', 'Set how many tries you get and how the tutor grades an answer.'),
              review=('Review', 'Set when an idea comes back for review.'),
              links=('Links', 'Set how much of a link the tutor reads.'),
              research=('Research', 'Set how the tutor finds a source on the web for an idea that has none.'),
              stt=('Speech to text', 'Pick the model that writes down what you say.'),
              voice=('Speaking voice', 'Pick the voice that reads the steps of the screen guide aloud.'),
              guide=('Screen guide', 'Set what the screen guide does when it explains a part of the page.'),
              models=('Models', 'Pick the model for each task. "Default" uses the default model.'),
              lists=('Lists', 'Set how many items show in the lists of the app.'),
              prompts=('Prompts', 'These texts tell the model how to ask, grade, hint and teach.'))

### The table

`add` puts one row in `SETTINGS`. The name is `section.key`: the section of `settings.json` and the key in it. The type comes from the default.
`more` puts the row under the closed "More" part of its group. `store='profile'` marks a value that the vault keeps in your profile, not this file.
`hi` can be the name of another row. Then that value is the top of the range. `ui=False` marks a value that its group shows with its own controls. `reset=False` keeps a value when you use the defaults of its group: the theme, and the speech engine and model.

In [ ]:
#| export
SETTINGS = {}   # 'section.key' -> the row

def add(name:str, group:str, label:str, help:str, dflt, lo=None, hi=None, more:bool=False, choices=None, unit:str='',
        pct:bool=False, store:str='file', ui:bool=True, reset:bool=True):
    "Add the setting `name` to `SETTINGS` and return its row."
    kind = ('bool' if isinstance(dflt, bool) else 'choice' if choices else 'int' if isinstance(dflt, int) else
            'float' if isinstance(dflt, float) else 'str')
    sec, key = name.split('.', 1)
    SETTINGS[name] = r = AttrDict(name=name, section=sec, key=key, group=group, label=label, help=help, kind=kind, dflt=dflt,
                                  lo=lo, hi=hi, more=more, choices=tuple(choices or ()), unit=unit, pct=pct, store=store, ui=ui, reset=reset)
    return r

The **Look** group keeps its own controls: the theme swatches and the mode. The **Learning** group holds the three values of your profile.

In [ ]:
#| export
add('look.theme', 'look', 'Theme', 'The colours of the app.', 'paper', ui=False, reset=False)
add('look.mode', 'look', 'Light or dark', 'Auto follows the Mac.', 'auto', ui=False, reset=False)
add('look.keys_float', 'look', 'Keys button', 'A small Keys button at the bottom left of each page. It shows the most useful keys of the page.', True)
add('learning.pace', 'learning', 'Pace', 'How big each step is. Gentle uses small steps. Fast uses bigger steps.', 'steady',
    choices=('gentle', 'steady', 'fast'), store='profile')
add('learning.session_len', 'learning', 'Questions in a session', 'How many questions the tutor asks in one session.', 12, 3, 40, store='profile')
add('learning.explore', 'learning', 'How often to try a new way', 'How often the tutor teaches in a way that you did not try yet.', 'sometimes',
    choices=('never', 'sometimes', 'often'), store='profile');

The **Walks** group sets the size of a walk and of a side walk.

In [ ]:
#| export
add('walks.steps', 'walks', 'Ideas in a walk', 'The most big ideas in one walk, from 3 to 7. The other ideas stay on the steps as reading, and in your concepts.', 7, 3, 7)
add('walks.hands', 'walks', 'Hands-on share', 'The share of checks in a walk that are a demo, an order or code. The others are short answers.', 1/3, 0., 1., pct=True)
add('walks.chain', 'walks', 'Ideas in a side walk', 'The most ideas in a side walk beside the paper.', 4, 2, 8)
add('walks.passage', 'walks', 'Passage length', 'The most characters of the source for each idea.', 600, 200, 2000, more=True, unit='characters')
add('walks.note', 'walks', 'Note length', 'The most words in the tutor note under the passage.', 60, 20, 120, more=True, unit='words')
add('walks.depth', 'walks', 'Side walk depth', 'How many side walks you can open, one inside the other. A deeper idea opens as its own session.', 2, 1, 4, more=True)
add('walks.fresh', 'walks', 'Time for "Back on the paper"', 'How long the "Back on the paper" line shows after a side walk.', 10, 1, 60, more=True, unit='min')
add('walks.demo_wait', 'walks', 'Wait for a demo', 'How long an idea waits for its demo. Then the check is a short answer.', 90, 30, 300, more=True, unit='s');

The **Talks** group sets how a talk runs: the questions, the time, the vote and the room.

In [ ]:
#| export
add('talks.choice', 'talks', 'Choice questions from short checks', 'A new talk makes each short check a question with 4 options. When this is off, the question asks for a short answer.', True)
add('talks.ask_min', 'talks', 'Minutes for each question', 'The minutes that a question adds to its slide in the time of a talk.', 1, 0, 5, unit='min')
add('talks.pair_min', 'talks', 'Time to talk to a neighbour', 'How long the room talks in pairs before a second vote.', 2, 1, 10, unit='min')
add('talks.weak', 'talks', 'Unsure below', 'A slide is unsure below this share of right answers. The recap lists the unsure slides.', .7, .3, .95, more=True, pct=True)
add('talks.split_lo', 'talks', 'A split vote starts at', 'A vote is split from this share of right answers to the end value. Then the room talks in pairs.', .3, 0., 1., more=True, pct=True)
add('talks.split_hi', 'talks', 'A split vote ends at', 'The top of a split vote. Above it, most of the room knows the answer.', .7, 0., 1., more=True, pct=True)
add('talks.laser_ms', 'talks', 'Laser fade', 'How long the laser dot stays on the audience screen after the pointer stops.', 1500, 500, 5000, more=True, unit='ms')
add('talks.image_mb', 'talks', 'Largest figure for the room', 'A figure larger than this does not go to the room.', 5, 1, 20, more=True, unit='MB');

The **Checks and grading** group sets the tries, the grade of a shaky pass, and when an idea counts as learned.

In [ ]:
#| export
add('checks.tries', 'checks', 'Tries', 'How many answers you can give before the tutor teaches the idea.', 3, 1, 5)
add('checks.sources', 'checks', 'Use my sources', 'The tutor makes its questions from what you read and wrote.', True)
add('checks.one_ask', 'checks', 'One ask per question', 'When a question asks two things, the tutor writes it again with one ask.', True, more=True)
add('checks.readable', 'checks', 'Plain English', 'When a text that the tutor writes is hard to read, the tutor writes it again in plain English, once.', True, more=True)
add('checks.near', 'checks', 'Shaky pass', 'A grade at or above this number is solid. A pass below it is shaky, and it comes back sooner.', 7, 5, 9, more=True)
add('checks.goal', 'checks', 'Learned level', 'The level at which an idea counts as learned. 2 is recall, 3 is explain, 4 is apply and 5 is transfer.', 4, 2, 5, more=True)
add('checks.high', 'checks', 'Sure confidence', 'At or above this confidence, you are sure. A sure answer that is wrong counts as a misconception.', 60, 40, 90, more=True, unit='%')
add('checks.hints', 'checks', 'Hints before a pass is shaky', 'With this many hints, a right answer counts as shaky.', 2, 1, 4, more=True)
add('checks.demo_misses', 'checks', 'Misses before a demo', 'After this many misses on an idea, the tutor makes a demo for it.', 2, 1, 5, more=True)
add('checks.demo_ladder', 'checks', 'More help after a demo', 'After a miss on a demo, the tutor teaches the idea in another way. After the next miss, it shows a simpler demo.', True, more=True)
add('checks.retest', 'checks', 'Retest a miss after', 'The tutor asks a missed idea again after this many other questions in the session.', 2, 1, 6, more=True, unit='questions')
add('checks.feedback_row', 'checks', 'Feedback under a demo or a lesson', 'A short row under each demo and lesson: Too simple, Just right or Not clear. The tutor uses your choice for the next demo or lesson.', True)
add('checks.open_days', 'checks', 'Keep an open check for', 'An open check with no answer for this many days closes. It gets no grade.', 7, 1, 60, more=True, unit='days');

The **Review** group sets the time to the next review. A daily limit on review is new: 0 means no limit, as before.

In [ ]:
#| export
add('review.solid', 'review', 'First review after a solid answer', 'When a new idea comes back after your first solid answer.', 1., .25, 7., unit='days')
add('review.per_day', 'review', 'Most reviews a day', 'The tutor stops review after this many questions in a day. 0 means no limit.', 0, 0, 200)
add('review.shaky', 'review', 'First review after a shaky answer', 'When a new idea comes back after your first shaky answer.', .5, .1, 3., more=True, unit='days')
add('review.growth', 'review', 'Growth factor', 'How fast the time to the next review grows after a solid answer. With a higher number, reviews come less often.', 3., 1., 6., more=True)
add('review.fading', 'review', 'Fading below recall', 'An idea is fading when the chance that you recall it is below this value.', .6, .3, .9, more=True, pct=True);

The **Links** group sets how much of a link the tutor gets. The pages read by default can not be more than the page cap.

In [ ]:
#| export
add('links.pages', 'links', 'Pages read by default', 'How many pages of a link the tutor reads to make a walk.', 8, 1, 'links.cap')
add('links.cap', 'links', 'Page cap', 'The most pages that the tutor reads from one link.', 20, 1, 50, more=True)
add('links.timeout', 'links', 'Time for each fetch', 'How long the tutor waits for one page. Then it stops.', 30, 5, 120, more=True, unit='s')
add('links.speed', 'links', 'Download speed cap', 'The fastest speed at which abhyasa gets a page.', 1., .1, 10., more=True, unit='MB/s')
add('links.kept', 'links', 'Characters kept for each page', 'The most characters of one page that your vault keeps.', 400_000, 10_000, 1_000_000, more=True)
add('links.read', 'links', 'Characters read for each page', 'The most characters of one page that the tutor reads for ideas.', 100_000, 10_000, 1_000_000, more=True);

The **Research** group sets the web search for a step with no source. The tutor searches the web for the idea, reads the best pages, and quotes one passage word for word. When research is off, the tutor does no web search, and the step says that the text is in the tutor's words.

In [ ]:
#| export
add('research.on', 'research', 'Find sources on the web', 'For an idea with no source, the tutor searches the web and quotes a page. The search engine gets your topic.', True)
add('research.per_walk', 'research', 'Ideas with a web source in a walk', 'The most ideas of one new walk that get a web source. 0 means none.', 12, 0, 30)
add('research.results', 'research', 'Search results to read', 'How many search results the tutor looks at for one idea. It reads the best 3 pages.', 5, 3, 10, more=True)
add('research.retry', 'research', 'Look again after', 'An idea that waits longer than this for a web source gets a new search. A restart can stop a search.', 10, 1, 120, more=True, unit='min')
add('research.agent', 'research', 'Let an agent gather sources for a big goal', 'An agent reads your sources and the web, and gives a passage for each idea. It works for a goal that names two or more sources or a big folder. abhyasa checks each passage word for word.', True)
add('research.agent_minutes', 'research', 'Most minutes for the agent', 'The agent stops after this time. You can also stop it.', 15, 1, 60, more=True, unit='min')
add('research.told', 'research', 'Web search line shown', 'The Start box said one time that a topic walk searches the web.', False, ui=False);
add('sources.roots', 'research', 'Source folders', 'Where abhyasa looks for a source that you name. It looks two levels down. It never searches the whole disk.', '~/code/personal/orgs');

The **Speech to text** group keeps the engine and the model, and the pauses at which the mic cuts what you say.
The **Screen guide** group has the voice and the look of a point mark. A point mark is weaker than a circle in each look,
so a circle is always the main mark of a step. The default is `tick`, because it has no ring that looks like a circle.

In [ ]:
#| export
add('stt.engine', 'stt', 'Engine', 'The speech engine.', 'mlx', ui=False, reset=False)
add('stt.model', 'stt', 'Model', 'The speech model. Empty uses the default of the engine.', '', ui=False, reset=False)
add('stt.segment', 'stt', 'Cut at pauses', 'The text shows after each pause. When this is off, the text comes in fixed blocks of time.', True)
add('stt.pause', 'stt', 'Pause before a cut', 'How long a pause must be before the text of what you said shows.', 700, 300, 2000, more=True, unit='ms')
add('stt.longest', 'stt', 'Longest segment', 'The text shows after this time, also when you do not pause.', 12, 5, 30, more=True, unit='s')
add('guide.voice', 'guide', 'Voice', 'The guide speaks each step aloud. When this is off, the words of each step light up with no sound.', True)
add('guide.point', 'guide', 'Point mark', 'How the guide points to a part of the page. Tick is a short pen tick. Dot is a small dot that pulses one time. Ring is a thin, small ring. Each look is weaker than a circle, so the part that you asked about stands out.', 'tick',
    choices=('tick', 'dot', 'ring'));

The **Speaking voice** group picks who reads the steps of the screen guide aloud. `pocket` runs Pocket TTS on this Mac. `openai` sends each line to OpenAI.
`browser` is the voice of the browser (or of the Mac in the overlay). The `voice` module reads these values.
The default is `pocket`. Until its model is downloaded, the browser speaks. An old `hf` or `kokoro` value is not one of the choices, so it reads as `pocket`.
The speed is only for OpenAI: Pocket TTS has no speed.

In [ ]:
#| export
add('voice.out', 'voice', 'Speaking engine', 'Who reads the steps aloud: Pocket TTS on this Mac, OpenAI, or the browser.', 'pocket',
    choices=('pocket', 'openai', 'browser'), ui=False, reset=False)
add('voice.out_model', 'voice', 'Voice model', 'The model of the speaking voice. Empty uses the default of the engine.', '', ui=False, reset=False)
add('voice.out_voice', 'voice', 'Voice', 'The voice of the model. Empty uses the default of the engine.', '', ui=False, reset=False)
add('voice.out_speed', 'voice', 'Speed', 'How fast the OpenAI voice speaks. 1 is the normal speed. The browser voice uses the speed of the board.', 1., .5, 2.);

The **Models** group has the default model and one model for each task. An empty value uses the default model.
The tasks after `extract` borrow the model of an older task when they have none: `BORROWS` says which. The screen guide (`guide`) and its figure labels (`guide_vision`) use the model for hints until you pick one. The builder chat (`build`) is an agent: it uses the default model until you pick one, and it has no reply cap. The reply caps are in tokens.

In [ ]:
#| export
TASK_HELP = dict(question='Writes each question and its hidden answer.', grade='Reads your answer and grades it.', hint='Gives the next hint.',
                 lesson='Teaches the smallest step after a miss.', extract='Finds the ideas in a text.', plan='Plans the ideas of a walk to a goal.',
                 pick='Picks the main ideas of a long source for a walk.', note='Writes the tutor note under the passage of an idea.',
                 profile='Learns about you from your answers.',
                 talk='Writes a choice question for a talk from a short check.',
                 guide='Explains a part of the page when you ask the screen guide.',
                 guide_vision='Reads the labels of a figure for the screen guide. Pick a model that can see pictures.',
                 build='Talks with you in the builder and changes the walk or the talk.')
BORROWS = dict(plan='extract', pick='extract', note='hint', profile='extract', talk='question', guide='hint', guide_vision='hint')
TASK_LABEL = dict(question='Questions', grade='Grades', hint='Hints', lesson='Lessons', extract='Ideas in a text', plan='Plans',
                  pick='Main ideas', note='Walk notes', profile='Your profile', talk='Talk questions',
                  guide='Screen guide', guide_vision='Figure labels', build='Builder chat')

add('models.default', 'models', 'Default model', 'The model for each task that has no model of its own. '
    '"abhyasa --model" and $ABHYASA_MODEL come first when they are set.', '', ui=False)
for t, h in TASK_HELP.items():
    add(f'models.{t}', 'models', TASK_LABEL[t], h + (f' Empty uses the model for {TASK_LABEL[BORROWS[t]].lower()}.' if t in BORROWS else ''),
        '', more=t not in ('question', 'grade', 'hint', 'lesson', 'guide', 'guide_vision'), ui=False)
for t, n in dict(question=4096, grade=2048, hint=512, lesson=2048, extract=4096).items():
    add(f'tokens.{t}', 'models', f'Reply cap: {TASK_LABEL[t].lower()}', 'The most tokens in one reply. Claude Code models do not use this limit.',
        n, 1, 32768, more=True, unit='tokens')

The **Lists** group sets how many items show in the lists of the app. All of it is under "More".

In [ ]:
#| export
add('lists.strengths', 'lists', 'Weak and strong ideas', 'How many weak spots and strong ideas the Strengths page shows.', 10, 3, 50, more=True)
add('lists.weeks', 'lists', 'Weeks in your history', 'How many weeks the history of your learning shows.', 8, 1, 52, more=True)
add('lists.search', 'lists', 'Search results', 'How many results from what you learned the search shows.', 12, 3, 50, more=True)
add('lists.elsewhere', 'lists', 'Results from other sources', 'How many results from your other sources the search shows.', 2, 0, 10, more=True)
add('lists.snippet', 'lists', 'Length of a search result', 'The most characters of each search result.', 300, 100, 2000, more=True, unit='characters')
add('lists.attempts', 'lists', 'Answers on a concept page', 'How many of your recent answers a concept page shows.', 10, 3, 50, more=True)
add('lists.changes', 'lists', 'Changes on the Profile page', 'How many changes to your profile the Profile page shows.', 40, 10, 200, more=True)
add('lists.sort_batch', 'lists', 'Concepts in one sort', 'How many concepts the tutor sorts into groups in one request. With fewer, each request is shorter.', 40, 10, 100, more=True);

Each group except **Prompts** has rows, and each row is in a group. The Prompts group lists the prompts of `prompts.PROMPTS`.

In [ ]:
assert set(GROUPS) - {'prompts'} == {r.group for r in SETTINGS.values()}
assert all(r.kind != 'int' or r.lo <= r.dflt for r in SETTINGS.values())
len(SETTINGS)

### The file

`DFLT_SETTINGS` is the default of each section: the rows that the file keeps, the saved prompt texts and the models that the Settings page downloaded.
`settings()` gives the saved settings over the defaults. A section that is not a dict in the file is ignored, and a top-level key that is not known is dropped.

In [ ]:
#| export
def cfg_dir() -> Path:
    "The folder for abhyasa's settings and keys: `$ABHYASA_CONFIG`, else `~/.config/abhyasa`."
    return Path(os.getenv('ABHYASA_CONFIG') or '~/.config/abhyasa').expanduser()

DFLT_SETTINGS = dict(prompts={}, downloaded=[])   # your text for a prompt, by name; the models that the Settings page downloaded
for _r in SETTINGS.values():
    if _r.store == 'file': DFLT_SETTINGS.setdefault(_r.section, {})[_r.key] = _r.dflt

A host sets context variables for one request, such as the settings of `use_settings` or the workspace of the caller. `CtxPool` is a thread pool whose jobs run in a copy of the context of the call that submitted them, so a job in the background sees the same values.

In [ ]:
#| export
from concurrent.futures import ThreadPoolExecutor

class CtxPool(ThreadPoolExecutor):
    "A thread pool whose jobs run in a copy of the submitter's context."
    def submit(self, fn, /, *a, **kw): return super().submit(contextvars.copy_context().run, fn, *a, **kw)

A value set before `submit` is seen by the job, also through `map`. A value set after is not.

In [ ]:
v = contextvars.ContextVar('v', default='none')
with CtxPool(1) as p:
    tok = v.set('mine')
    test_eq(p.submit(v.get).result(), 'mine')
    test_eq(list(p.map(lambda _: v.get(), [0])), ['mine'])
    v.reset(tok)
    test_eq(p.submit(v.get).result(), 'none')

`_saved` reads the file only when it changed, so a value read in a loop costs one `stat`.

In [ ]:
#| export
_cache = {}
_given = contextvars.ContextVar('given_settings', default=None)   # the rows a host passes for one call; None means read the file

def _path() -> Path: return cfg_dir()/'settings.json'

def _saved() -> dict:
    "The saved file as a dict; empty when it is missing or bad. It is read again only when it changed. In `use_settings` it is the given rows, and the file is not read."
    if (g := _given.get()) is not None: return g
    p = _path()
    try: st = p.stat()
    except OSError: return {}
    k = (str(p), st.st_ino, st.st_mtime_ns, st.st_size)
    if _cache.get('k') != k:
        try: d = json.loads(p.read_text())
        except (OSError, ValueError): d = {}
        _cache.update(k=k, d=d if isinstance(d, dict) else {})
    return _cache['d']

def settings() -> dict:
    "The saved settings over the defaults."
    d = json.loads(json.dumps(DFLT_SETTINGS))
    for k, v in _saved().items():
        if isinstance(d.get(k), dict):
            if isinstance(v, dict): d[k].update(v)
        elif k in d and type(v) is type(d[k]): d[k] = v
    return json.loads(json.dumps(d))

`save_settings` writes the file: first a temporary file with its own name, then a rename, so a crash never leaves half a file. A lock keeps two changes from the web app apart. It does not keep a value that is the default. `update_settings` changes keys in one section.

In [ ]:
#| export
_lock = threading.RLock()   # the web app saves from many threads: one read-change-write at a time

def save_settings(d:dict) -> dict:
    "Write all the settings. In `use_settings` the host owns the values, so a write is refused."
    if _given.get() is not None: raise RuntimeError('settings are given by the host for this call; abhyasa does not write its settings file')
    with _lock: return _write(d)

def _write(d:dict) -> dict:
    p = _path()
    p.parent.mkdir(parents=True, exist_ok=True)
    lean = {k: dict(v) if isinstance(v, dict) else v for k, v in d.items()}
    for r in SETTINGS.values():   # a default is not kept, so a new default in a later version works
        sec = lean.get(r.section)
        if isinstance(sec, dict) and r.key in sec and type(sec[r.key]) is type(r.dflt) and sec[r.key] == r.dflt: del sec[r.key]
    tmp = p.with_suffix(f'.{os.getpid()}.{threading.get_ident()}.tmp')   # its own name, so two writes never share a file
    tmp.write_text(json.dumps(lean, indent=2)); tmp.replace(p)
    _cache.clear()
    return d

def update_settings(section:str, **kw) -> dict:
    "Change some keys in one section of the settings."
    with _lock:
        d = settings(); d.setdefault(section, {}).update(kw)
        return save_settings(d)

With no file, `settings()` gives the defaults. A change in one section keeps the others.

In [ ]:
test_eq(settings()['walks']['steps'], 7)
update_settings('stt', segment=False)
test_eq(settings()['stt']['segment'], False)
test_eq(settings()['look'], dict(theme='paper', mode='auto', keys_float=True))
update_settings('stt', segment=True)

A file from before this table still loads: its sections update the defaults, and the new sections get their defaults.

In [ ]:
_path().write_text(json.dumps(dict(stt=dict(engine='mlx', model='m', segment=True), models=dict(question='', grade='', hint='', lesson=''),
                                   tokens=dict(question='', grade='', hint='', lesson='', extract=''), look=dict(theme='ink', mode='dark'), prompts={}, downloaded=[])))
d = settings()
test_eq((d['stt']['engine'], d['stt']['pause'], d['look']['theme'], d['walks']['steps'], d['models']['extract']), ('mlx', 700, 'ink', 7, ''))
save_settings(dict(prompts={}, downloaded=[]))

A file that is not JSON, or a section that is not a dict, gives the defaults. An unknown top-level key is dropped.

In [ ]:
_path().write_text('{not json')
test_eq(settings()['walks']['steps'], 7)
_path().write_text(json.dumps(dict(walks=5, mine=1)))
d = settings()
test_eq((d['walks']['steps'], 'mine' in d), (7, False))
save_settings(dict(prompts={}, downloaded=[]))

### Read one value

`fit` makes a saved value fit its row. A number outside the range gets the nearest end. A value of the wrong type gets the default.
A whole number can come as `12.0` from a form; it stays a whole number. A choice must be one of the choices.

In [ ]:
#| export
def top(r):
    "The top of the range of row `r`: a number, or the value of the row that `r.hi` names."
    return val(r.hi) if isinstance(r.hi, str) else r.hi

def fit(r, v):
    "The value `v` made to fit row `r`: clamped to the range, or the default for the wrong type."
    if r.kind == 'bool': return v if isinstance(v, bool) else r.dflt
    if r.kind == 'choice': return v if v in r.choices else r.dflt
    if r.kind == 'str': return v if isinstance(v, str) else r.dflt
    if isinstance(v, str):   # a number that you typed in the file by hand, such as "8192"
        try: v = float(v.strip())
        except ValueError: return r.dflt
    if isinstance(v, bool) or not isinstance(v, (int, float)) or v != v: return r.dflt
    v = min(max(v, r.lo), top(r))
    return int(round(v)) if r.kind == 'int' else float(v)

`val` gives the value of a row now: the saved value made to fit, else the default. The engine calls it each time it needs the value.

In [ ]:
#| export
def raw(name:str):
    "The saved value of `name` as it is in the file, or None."
    r = SETTINGS[name]
    sec = _saved().get(r.section)
    return sec.get(r.key) if isinstance(sec, dict) else None

def val(name:str):
    "The value of the setting `name` now."
    r = SETTINGS[name]
    if r.store != 'file': raise KeyError(f'{name} is kept in the profile; read it from `Learning.profile()`')
    v = raw(name)
    return r.dflt if v is None else fit(r, v)

def dflt(name:str): return SETTINGS[name].dflt

`use_settings` gives a host its own values for the rows that the file keeps, for the code inside one `with`. The host passes `rows`: the row name (`'walks.steps'`) to its value. A name that is not a file row is dropped. Each value still goes through `fit`, so a number outside the range or a value of the wrong type is made to fit. A row that the host does not give has its default, and the file is not read. A context variable holds the rows, so another thread, or a later call, does not see them. `rows=None` changes nothing.

In [ ]:
#| export
@contextmanager
def use_settings(rows:dict|None):
    "Read `rows` (row name to value) instead of the settings file, for this call only; `None` keeps what is in use."
    if rows is None: yield; return
    given = {}
    for n, v in rows.items():
        if (r := SETTINGS.get(n)) and r.store == 'file': given.setdefault(r.section, {})[r.key] = v
    tok = _given.set(given)
    try: yield
    finally: _given.reset(tok)

A given value is used inside the `with` and not after it. It is made to fit its row, and a name that is not a row is dropped. A saved value in the file does not show inside the `with`.

In [ ]:
update_settings('walks', steps=5)
with use_settings({'walks.steps': 99, 'walks.note': 30, 'nope.x': 1, 'prompts.x': 2}):
    inside = (val('walks.steps'), val('walks.note'), val('walks.chain'), settings()['walks']['steps'])
with use_settings(None): none = val('walks.steps')
update_settings('walks', steps=7)
test_eq((inside, none, val('walks.steps')), ((7, 30, dflt('walks.chain'), 99), 5, 7))   # `val` fits a value to its row; `settings()` keeps the raw value, as it does for the file

`sources.roots` is a text list, one path for each line. A source folder has the same rules as a folder that you name: it
must be in your home folder, it must not be the home folder itself, and no part of it can be hidden or a link. `source_root_why`
gives the reason when a path breaks a rule. `roots` gives the paths that keep the rules, and `dropped_roots` gives the
others with their reasons, so the Settings page can show them.

In [ ]:
#| export
NO_FOLDER = 'abhyasa cannot find this folder.'

def source_root_why(p) -> str:
    "Why abhyasa does not read the source folder `p`: outside the home folder, the home folder itself, hidden, or a link. '' when it can."
    p, home = Path(str(p)).expanduser(), Path.home().resolve()
    try: r = p.resolve()
    except (OSError, RuntimeError): return NO_FOLDER
    if not r.is_relative_to(home): return 'abhyasa reads only folders in your home folder.'
    if r == home: return 'Name a folder in your home folder, not the home folder itself.'
    if p.is_symlink() or any(x.startswith('.') for x in r.relative_to(home).parts): return 'abhyasa does not read a hidden folder or a link.'
    return ''

def _root_lines(): return [l.strip() for l in str(val('sources.roots') or '').splitlines() if l.strip()]

def roots() -> list:
    "The paths of the setting `sources.roots` that keep the rules of `source_root_why`, one for each line that is not empty."
    return [l for l in _root_lines() if not source_root_why(l)]

def dropped_roots() -> list:
    "The lines of `sources.roots` that abhyasa does not use, each as (line, reason)."
    return [(l, w) for l in _root_lines() if (w := source_root_why(l))]

In [ ]:
test_eq(roots(), ['~/code/personal/orgs'])

A saved 50 ideas becomes 7, the most. A string gives the default. A share above 1 becomes 1.

In [ ]:
update_settings('walks', steps=50, hands='a lot', note=40.0)
test_eq((val('walks.steps'), val('walks.hands'), val('walks.note')), (7, 1/3, 40))
update_settings('walks', hands=1.5, steps=1)
test_eq((val('walks.hands'), val('walks.steps')), (1., 3))
save_settings(dict(prompts={}, downloaded=[]))

The top of the pages read by default is the page cap.

In [ ]:
update_settings('links', pages=40, cap=10)
test_eq((val('links.pages'), val('links.cap')), (10, 10))
save_settings(dict(prompts={}, downloaded=[]))

The look of a point mark is one of three choices. A look that is not a choice gives the default, `tick`.

In [ ]:
update_settings('guide', point='dot')
test_eq(val('guide.point'), 'dot')
update_settings('guide', point='arrow')
test_eq((val('guide.point'), SETTINGS['guide.point'].choices), ('tick', ('tick', 'dot', 'ring')))
save_settings(dict(prompts={}, downloaded=[]))

### Change one value

`set_val` saves a value made to fit its row, and returns it. `reset` puts back the default of one row, and `reset_group` of each row in a group.
`changed` is true when a value is not its default. `clamped` gives the value in the file when it was outside the range, so the page can say so.

In [ ]:
#| export
def set_val(name:str, v):
    "Save the value `v` of the setting `name`, made to fit its row. Return the value in use."
    r = SETTINGS[name]
    update_settings(r.section, **{r.key: fit(r, v)})
    return val(name)

def reset(name:str):
    "Use the default of the setting `name` again."
    r = SETTINGS[name]
    if r.store != 'file': raise KeyError(f'{name} is kept in the profile; change it with `Learning.set_profile()`')
    with _lock:
        d = settings(); d[r.section].pop(r.key, None)
        save_settings(d)
    return r.dflt

def rows(group:str) -> list:
    "The rows of `group`, in table order."
    return [r for r in SETTINGS.values() if r.group == group]

def reset_group(group:str) -> list:
    "Use the default of each row of `group` that the file keeps and that a group reset can change. Return their names."
    ns = [r.name for r in rows(group) if r.store == 'file' and r.reset and raw(r.name) is not None]
    with _lock:
        d = settings()
        for n in ns: d[SETTINGS[n].section].pop(SETTINGS[n].key, None)
        save_settings(d)
    return ns

def changed(name:str) -> bool: return val(name) != dflt(name)

def clamped(name:str):
    "The number in the file for `name` when it was outside the range, else None."
    r, v = SETTINGS[name], raw(name)
    if r.kind not in ('int', 'float') or isinstance(v, bool) or not isinstance(v, (int, float)): return None
    return v if v < r.lo or v > top(r) else None

The source folders drop the root of the disk, the home folder itself and a hidden folder.

In [ ]:
set_val('sources.roots', '/\n~\n~/.ssh\n~/code/personal/orgs\n')
test_eq(roots(), ['~/code/personal/orgs'])
test_eq([l for l, _ in dropped_roots()], ['/', '~', '~/.ssh'])
reset('sources.roots')

A value saved out of range is saved as the end of the range. `reset` and `reset_group` put back the defaults.

In [ ]:
test_eq(set_val('walks.steps', 99), 7)
test_eq(set_val('walks.steps', 1), 3)
test_eq(raw('walks.steps'), 3)
assert changed('walks.steps') and not changed('walks.note')
reset('walks.steps')
test_eq(val('walks.steps'), 7)
set_val('checks.tries', 5); set_val('checks.near', 8)
test_eq(sorted(reset_group('checks')), ['checks.near', 'checks.tries'])
test_eq((val('checks.tries'), val('checks.near')), (3, 7))
update_settings('walks', steps=50)
test_eq(clamped('walks.steps'), 50)
# a number typed by hand as text works, and text that is not a number gives the default
update_settings('tokens', hint='900', grade='many')
test_eq((val('tokens.hint'), val('tokens.grade')), (900, 2048))
# a value that the profile keeps has no reset here
test_fail(lambda: reset('learning.pace'), contains='profile')
save_settings(dict(prompts={}, downloaded=[]))

The speaking engine is `browser` until you change it, and a value that is not an engine gives the default.

In [ ]:
test_eq((val('voice.out'), val('voice.out_speed')), ('pocket', 1.))
test_eq((fit(SETTINGS['voice.out'], 'kokoro'), fit(SETTINGS['voice.out'], 'hf')), ('pocket', 'pocket'))

### What stays fixed

Some values keep you safe, so they have no setting: the link checks for local addresses, plain HTTP, the link kinds, the HTML allowlist, the check of a pane address, the local-only hosts, the test allowlist, the grade scale and the depth ladder.
The room for phones keeps its safety limits fixed too: the bind rule, the join code, the rate limits, the body size and the headers.
`FIXED` names them, and no row may have one of these words in its name.

In [ ]:
#| export
FIXED = ('network', 'sandbox', 'audit', 'isolate', 'local', 'plain', 'link_kind', 'html', 'pane_url', 'hosts', 'allowlist', 'safe_names', 'grade_scale', 'depths',
         'bind', 'join_code', 'rate', 'body', 'room_headers')

In [ ]:
assert not [n for n in SETTINGS for w in FIXED if w in n]

## Prompts

The instruction text of the tutor. You can change it on the Settings page.

Each prompt of the tutor has two parts. The variable parts (the concept, the thread, the JSON shape) stay in code. The instruction text is in this registry.
Each prompt has a name, a plain label, a note on what it changes, and its default text from code. The Settings page shows each prompt, and you can change its text there.
The tutor reads each prompt when it calls the model. Thus a change works on the next question, with no restart.

Import the tools that this module uses. The settings come from `settings` only when a prompt is read, so the import stays light.

### The registry

`register` adds a prompt to `PROMPTS` and returns its default text. A `check` can refuse a bad text: it raises `ValueError` with a plain message.

In [ ]:
#| export
PROMPTS = {}   # name -> the prompt: name, label, note, default text and check

def register(name:str, label:str, note:str, dflt:str, check=None) -> str:
    "Add the prompt `name` with a plain `label`, a `note` on what it changes, its `dflt` text and an optional `check`. Return `dflt`."
    PROMPTS[name] = AttrDict(name=name, label=label, note=note, dflt=dflt, check=check)
    return dflt

`lines` gives the lines of a text that are not empty. `need_lines` makes a check that refuses a text without exactly `n` such lines.

In [ ]:
#| export
def lines(text:str) -> list:
    "The lines of `text` that are not empty, without spaces at the ends."
    return [l.strip() for l in (text or '').splitlines() if l.strip()]

def need_lines(n:int, what:str):
    "A check that refuses a text that does not have `n` lines that are not empty."
    def check(text):
        if (k := len(lines(text))) != n: raise ValueError(f'{what} needs {n} lines, one on each line. This text has {k}.')
    return check

### Read and save

`prompt` gives the saved text of a prompt, else its default. `set_prompt` saves a text in the settings, under the `prompts` key. An empty text, or a text that is the same as the default, removes the saved text. Then the default is in use again, also when a new version changes the default.

In [ ]:
#| export
def saved() -> dict:
    "The saved prompt texts: name -> text."
    return settings().get('prompts') or {}

def prompt(name:str) -> str:
    "The text of the prompt `name` now: the saved text, else the default."
    t = saved().get(name)
    return t if isinstance(t, str) and t.strip() else PROMPTS[name].dflt

def set_prompt(name:str, text:str='') -> str:
    "Save the text of the prompt `name`. An empty text uses the default. A text that fails the check raises `ValueError`. Return the text in use."
    p, text = PROMPTS[name], (text or '').replace('\r\n', '\n').strip()
    if text and p.check: p.check(text)
    d = settings()
    ps = d.setdefault('prompts', {})
    if text and text != p.dflt.strip(): ps[name] = text
    else: ps.pop(name, None)
    save_settings(d)
    return prompt(name)

### Tests

A small prompt with a check of 2 lines. Its default is in use at first.

In [ ]:
register('greet', 'Greeting', 'The first line.', 'Say hello.\nSay your name.', need_lines(2, 'The greeting'))
test_eq(prompt('greet'), 'Say hello.\nSay your name.')

A saved text is in use at once, and the settings keep it under `prompts`.

In [ ]:
test_eq(set_prompt('greet', 'Say hi.\r\nSay your job.\n'), 'Say hi.\nSay your job.')
test_eq((prompt('greet'), saved()), ('Say hi.\nSay your job.', {'greet': 'Say hi.\nSay your job.'}))

A text that fails the check is refused with a plain message, and the saved text stays.

In [ ]:
test_fail(lambda: set_prompt('greet', 'Only one line.'), contains='needs 2 lines')
test_eq(prompt('greet'), 'Say hi.\nSay your job.')

An empty text, or only spaces, removes the saved text, so the default is in use again. A text that is the same as the default is not kept.

In [ ]:
test_eq(set_prompt('greet', '  \n '), 'Say hello.\nSay your name.')
set_prompt('greet', 'Say hello.\nSay your name.')
test_eq(saved(), {})

An unknown name is an error. Then the test prompt goes away.

In [ ]:
test_fail(lambda: prompt('nope'))
del PROMPTS['greet']

## The store

The `learning` shelf: concepts, attempts and sessions as vault documents.

This module keeps all learning data in your vault, on one shelf named `learning`.
Each concept, attempt and session is one vault document. Thus each tool that reads the vault can search it.
The other modules read and write the shelf only through the `Learning` class in this module.

Import the tools that this module uses, and the `Vault` class of the vault.

In [ ]:
#| export
import json, logging, re, threading, time, uuid, unicodedata
from fastcore.utils import *
from vishalakshi import Vault
from litesearch import fold_text

log = logging.getLogger(__name__)

### The shelf

Search reads the text of a document. The `meta` JSON keeps the state.
`Vault.set_meta` changes the state and does not embed the text again.
The table gives the source URI of each kind of document.

| kind | source |
|---|---|
| `concept` | `learning://concept/<slug>` |
| `attempt` | `learning://attempt/<id>` |
| `session` | `learning://session/<id>` |

These constants name the shelf, give the start state of a concept, and list the fields that merge.

In [ ]:
#| export
SHELF = 'learning'
#: the learner state of a concept that has no attempts yet
STATE = dict(floor=0, ceiling=6, stability=0., due=None, last=None, streak=0, misses=0, seen=0, shaky=None, misconceptions=[])
#: content fields that a new source adds to, and does not replace
LISTS = ('prereqs', 'related', 'aliases', 'provenance', 'questions')
_SKIP = ('content_hash', 'pii_scanned')

def meta_of(r) -> dict:
    "The meta of the document row `r`: a dict from vishalakshi `by_source`, or JSON text from `t.docs`."
    m = r['meta']
    return m if isinstance(m, dict) else json.loads(m or '{}')

`slugify` makes the key of a concept from its name. It changes a name in an Indic script, such as Devanagari, to IAST first. It changes `+`, `#` and `&` to words, so C, C++ and C# get different slugs. A name with no letters that it can read gets a short hash of the name. Thus the same name always gives the same slug. `_src` makes the source URI of a document.

In [ ]:
#| export
from vishalakshi.core import slug

def slugify(s:str) -> str:
    "Make a lower-case ASCII slug from `s` (vishalakshi `slug`, whole): `saṃdhi` and `संधि` give `samdhi`, and `C++` gives `c-plus-plus`."
    return slug(s, n=None, translit=True, symbols=True)

def idea_name(name) -> bool:
    "Whether `name` can name a new idea: it has a letter or a digit, and it is not one lower-case letter such as `x`. One capital letter, such as `C` or `R`, can."
    n = str(name or '').strip()
    return any(ch.isalnum() for ch in n) and not (len(n) == 1 and n.islower())

def real_idea(name, statement) -> bool:
    "Whether a new concept `name` with `statement` and no source text is an idea: an `idea_name`, and a statement of 3 words or more. A plan or a gap with no source uses it."
    return idea_name(name) and len(str(statement or '').split()) >= 3

def listed(c) -> bool:
    "Whether concept `c` shows in Library and the graph: an `idea_name`, and not a statement of one word. The stray `x` of an old tool had the statement `x`."
    return idea_name(c.get('name')) and len(str(c.get('statement') or '').split()) != 1

def usable_name(name) -> bool:
    "Whether `name` has two characters at least, and a letter or a digit. A name such as `x` is not an idea."
    n = str(name or '').strip()
    return len(n) >= 2 and any(ch.isalnum() for ch in n)

def _src(kind, slug): return f'learning://{kind}/{slug}'

`_union` merges two lists and keeps the order. `concept_text` makes the text that search reads.

In [ ]:
#| export
def _union(a, b):
    "Items of `a`, then the new items of `b`, in order, compared as JSON."
    seen, out = set(), []
    for x in (*(a or ()), *(b or ())):
        k = json.dumps(x, sort_keys=True, default=str)
        if k not in seen: seen.add(k); out.append(x)
    return out

def concept_text(c) -> str:
    "The searchable text of a concept: its name, statement, aliases and subject."
    al = ', '.join(c.get('aliases') or [])
    return '\n\n'.join(o for o in (c['name'], c.get('statement', ''), al and f'Also: {al}',
                                   c.get('subject') and f'Subject: {c["subject"]}') if o)

`Learning` opens one shelf of a vault. It also accepts a path, or `None` for the default vault.

In [ ]:
#| export
class Learning:
    "Concepts, attempts and sessions on one shelf of your vault."
    def __init__(self, vault=None, shelf:str=SHELF):
        if not isinstance(vault, Vault): vault = Vault(vault)
        self.vault, self.v = vault, vault.shelf(shelf)
        self._lock = threading.RLock()   # a change of a group is one step: look, then write
        try: self.repair_names()   # a name that is a slug becomes words, one time
        except Exception as e: log.warning('abhyasa could not repair the names of the concepts: %r', e)
        try: import abhyasa.path; self.repair_titles()   # a walk titled with an arXiv id gets the title of its paper, one time
        except Exception as e: log.warning('abhyasa could not repair the titles of the walks: %r', e)
        try: getattr(self, 'repair_subjects', lambda: 0)()   # the `journey` module adds it: a walk gives its subject to its concepts, one time
        except Exception as e: log.warning('abhyasa could not repair the subjects of the concepts: %r', e)

    def _docs(self, kind, where='', args=()):
        w = 'kind=?' + (f' and ({where})' if where else '')
        return self.v.t.docs(where=w, where_args=[kind, *args])

    def _row(self, r):
        m = {k: v for k, v in meta_of(r).items() if k not in _SKIP}
        return AttrDict({**STATE, **m, 'doc_id': r['id'], 'name': m.get('name', r['title'])})

    def __repr__(self): return f'Learning({self.v.path!r}: {len(self._docs("concept"))} concepts)'

### Shelves of documents

A vault file holds more stores than the shelves of documents. For example, the vault keeps the names that it finds in a store named `entities`. That store has no `doc_id` column and no `docs` table. vishalakshi's `Vault.shelf_names` gives only the shelves that hold documents. `doc_shelf` opens one of them with `shelf(create=False)`, or gives None. It never makes a new shelf, so a name in a link or in an old walk does not add a store to your vault.

In [ ]:
#| export
@patch
def doc_shelf(self:Learning, name:str):
    "The shelf of documents `name` (vishalakshi `shelf(create=False)`), or None when the vault has no such shelf. It never makes a shelf."
    if name in (self.vault.name, self.v.name): return self.vault if name == self.vault.name else self.v
    return self.vault.shelf(name, create=False) if name else None

### Read concepts

Each method returns concepts as `AttrDict`s with the learner state filled in.

In [ ]:
#| export
@patch
def concept(self:Learning, slug:str) -> AttrDict|None:
    "One concept by slug, with its learner state."
    r = self.v.by_source(_src('concept', slug))
    return self._row(r) if r else None

@patch
def concept_by_doc(self:Learning, doc_id:str) -> AttrDict|None:
    "The concept that a search hit (its `doc_id`) belongs to, or None when the hit is not a concept."
    r = first(self.v.t.docs(where="id=? and kind='concept'", where_args=[doc_id]))
    return self._row(r) if r else None

@patch
def concepts(self:Learning, subject:str=None) -> L:
    "All concepts, or the concepts of one subject."
    rs = self._docs('concept', *(("json_extract(meta, '$.subject')=?", [subject]) if subject else ()))
    return L(rs).map(self._row)

@patch
def subjects(self:Learning) -> list:
    "The subjects that have concepts, with counts."
    return self.v.db.q(f"select json_extract(meta, '$.subject') as subject, count(*) as n from [{self.v.t.prefix}docs] "
                       "where kind='concept' group by 1 order by 2 desc")

### Write concepts

`put` adds a concept or adds to it. It keeps the learner state, merges the list fields and replaces the other fields.
It does not change the group of a concept that the learner moved by hand (see Groups below).
Two fields say what the learner must understand. `core` is the understanding in 1 or 2 sentences: what the idea does and why. It has no sizes, years or scores, unless the number is the idea. `wrong` lists 1 to 3 common wrong ideas, each as `{'think': ..., 'fact': ...}`. The question and the grade work from these two fields. `statement` stays as the reference text. An empty `core` or `wrong` keeps the old value. `wrong_ideas` makes each wrong idea a dict.
`clean_name` makes a name one line with no control characters, at most `NAME_LEN` characters. `put` and `rename` use it.

In [ ]:
#| export
NAME_LEN = 200   # the most characters in the name of a concept

def clean_name(s, n:int=NAME_LEN) -> str:
    "`s` as one line of at most `n` characters, with no control or format characters. The joiners of Indic scripts stay."
    return ' '.join(''.join(' ' if unicodedata.category(ch)[0] == 'C' and ch not in '\u200c\u200d' else ch for ch in str(s or '')).split())[:n].strip()

def wrong_ideas(xs, n:int=3) -> list:
    "At most `n` wrong ideas from `xs`, each as `{'think': str, 'fact': str}`. A plain string is a `think` with no `fact`; an empty one goes."
    out = [dict(think=str(x.get('think') or '').strip(), fact=str(x.get('fact') or '').strip()) if isinstance(x, dict) else dict(think=str(x).strip(), fact='') for x in xs or ()]
    return [x for x in out if x['think']][:n]

@patch
def put(self:Learning,
        name:str,              # the concept's name
        statement:str='',      # a true, self-contained statement of the idea
        subject:str='',        # e.g. 'sanskrit', 'jyotisha', 'ml'
        slug:str=None,         # defaults to `slugify(name)`
        prereqs=(),            # slugs this concept needs first
        related=(),            # slugs linked to it, in any direction
        aliases=(),            # other names
        provenance=(),         # where it came from: dicts such as `{'src': 'pasted', 'at': ...}`
        questions=(),          # ready questions: `{'q':..., 'a':..., 'depth':...}`
        core:str='',           # the understanding: what the idea does and why, in 1 or 2 sentences
        wrong=(),              # 1 to 3 common wrong ideas: `{'think':..., 'fact':...}`
        **meta) -> AttrDict:
    "Add a concept or add to one. Keep the learner state, merge the list fields and replace the other fields. A name that you set by hand stays."
    with self._lock:   # one step: look at the old concept, then write
        name = clean_name(name)
        slug = slug or slugify(name)
        old = self.concept(slug) or {}
        if not old and not idea_name(name): raise ValueError(f'{name!r} is not the name of an idea. A name has a letter or a digit, and it is not one small letter such as x.')   # an old tool filed `x`
        if old.get('named_by') == 'hand' and meta.get('named_by') != 'hand': name = old['name']   # a name that you set stays: use `rename`
        if old.get('group_by') in ('hand', 'sort', 'placed', 'undone'): meta.pop('group', None); meta.pop('group_by', None)   # a group that you, a saved sort or a placement set stays: use `set_group`
        elif 'group' in meta: meta['group'] = clean_group(meta['group'], subject or old.get('subject', ''))
        new = dict(name=name, statement=statement or old.get('statement', ''), subject=subject or old.get('subject', ''), slug=slug,
                   core=' '.join(str(core or '').split()) or old.get('core', ''), wrong=wrong_ideas(wrong) or old.get('wrong') or [], **meta)
        for k, v in dict(prereqs=prereqs, related=related, aliases=aliases, provenance=provenance, questions=questions).items():
            new[k] = _union(old.get(k), v)
        new['prereqs'] = [p for p in new['prereqs'] if p != slug]
        m = {**STATE, **{k: v for k, v in old.items() if k != 'doc_id'}, **new}
        r = self.v.put(concept_text(m), title=name, source=_src('concept', slug), kind='concept', meta=m)
        return self.concept(slug)

`update` and `link` change one concept. `drop` removes it.

In [ ]:
#| export
@patch
def update(self:Learning, slug:str, **kv) -> AttrDict:
    "Change the state of a concept and do not embed its text again."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    self.v.set_meta(c.doc_id, **kv)
    return self.concept(slug)

@patch
def link(self:Learning, slug:str, prereq:str):
    "Record that `prereq` comes before `slug`."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    if prereq != slug and prereq not in c.prereqs: self.update(slug, prereqs=[*c.prereqs, prereq])

@patch
def drop(self:Learning, slug:str):
    "Remove a concept. Its attempts stay as a record of what happened."
    if (c := self.concept(slug)): self.v.forget(c.doc_id)

### Move old slugs

abhyasa 0.0.5 and older made slugs in a different way. A name in Devanagari got a random slug, and C, C++ and C# got the same slug, `c`.
`reslug` finds each concept that has an old slug. It moves the concept to the slug that `slugify` gives now. It also changes the slug
in each place that points to it: the links of other concepts, attempts, sessions, walks and demos. A slug that a source sets,
such as `sa-term-karaka`, does not change.

In [ ]:
#| export
SLUG_KEYS = ('concept', 'slug', 'focus', 'prereqs', 'related', 'concepts', 'split', 'gaps', 'of')

def _old_slug(s:str) -> str:
    "The slug that abhyasa 0.0.5 made from `s`; empty where it used a random one."
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]+', '-', s.lower()).strip('-')

def _swap(x, moves:dict, key=None):
    "`x` with each slug under a key in `SLUG_KEYS` changed as `moves` says."
    if isinstance(x, dict): return {k: _swap(v, moves, k) for k, v in x.items()}
    if isinstance(x, list): return [_swap(v, moves, key) for v in x]
    return moves.get(x, x) if key in SLUG_KEYS and isinstance(x, str) else x

`reslug` finds the moves first. With `dry_run` it stops there and writes nothing.

In [ ]:
#| export
@patch
def reslug(self:Learning, dry_run:bool=False) -> dict:
    "Move each concept with an old-style slug to the slug from `slugify`, and every slug that points to it. Returns `{old: new}`."
    cs = self.concepts()
    have, moves = set(cs.attrgot('slug')), {}
    for c in cs:
        old = _old_slug(c.name)
        if not (c.slug == old or (not old and re.fullmatch(r'[0-9a-f]{8}', c.slug))) or (new := slugify(c.name)) == c.slug: continue
        base, i = new, 2
        while new in have: new, i = f'{base}-{i}', i + 1
        moves[c.slug] = new; have.add(new)
    if dry_run or not moves: return moves
    for c in cs.filter(lambda c: c.slug in moves):
        m = {**json.loads(first(self.v.t.docs(where='id=?', where_args=[c.doc_id]))['meta'] or '{}'), 'slug': moves[c.slug]}
        self.v.add(concept_text(m), title=c.name, source=_src('concept', moves[c.slug]), kind='concept', meta=m)
        self.v.forget(c.doc_id)
    for r in self.v.t.docs():
        m = meta_of(r)
        if (n := _swap(m, moves)) != m: self.v.set_meta(r['id'], **{k: v for k, v in n.items() if v != m.get(k)})
    return moves

### Groups

A concept can sit in a group under its subject. For example, Magha sits at jyotisha › nakshatras › Magha.
The path is a list of at most two short names in the `group` field. `group_by` tells who set it: `hand` for the learner, `model` for the tutor.
The tutor never moves a concept that the learner moved by hand. An old vault has no groups, so each concept shows under "Not sorted yet".

`clean_group` makes a path safe to keep and to show. Each name has only letters, digits, spaces and hyphens, and at most `GROUP_LEN` characters.

In [ ]:
#| export
GROUP_DEPTH, GROUP_LEN = 2, 40   # the most levels in a path, and the most characters in one name

def _plain(s:str) -> str:
    "`s` with each character that is not a letter, a digit, a mark, a space or a hyphen made a space."
    return ''.join(ch if ch.isalnum() or ch in ' -' or unicodedata.category(ch)[0] == 'M' else ' ' for ch in s)

def clean_group(path, subject:str='') -> list:
    "`path` as at most `GROUP_DEPTH` short plain names. A text splits at `/`, `>` or `›`. Empty parts, repeats and a first part that is the subject drop."
    if isinstance(path, str): path = re.split(r'[/>›]', path)
    if not isinstance(path, (list, tuple)): return []
    out = []
    for p in path:
        s = ' '.join(_plain(p).split())[:GROUP_LEN].strip(' -') if isinstance(p, str) else ''
        if not s or s.lower() in [o.lower() for o in out] or (not out and s.lower() == (subject or '').lower()): continue
        out.append(s)
    return out[:GROUP_DEPTH]

`set_group` moves one concept. A move by hand marks it. A move by the model does not change a concept with the mark. An empty path by hand also keeps the mark: the learner chose no group, so the tutor does not sort the concept again.
`groups` lists the paths in use under one subject, with their parents, so a new concept can go in a group that is there.

In [ ]:
#| export
@patch
def set_group(self:Learning, slug:str, path, by:str='hand') -> AttrDict:
    "Put concept `slug` in the group `path`, set `by` the learner (`hand`) or the tutor (`model`)."
    with self._lock:
        c = self.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        if by != 'hand' and c.get('group_by') == 'hand': return c
        g = clean_group(path, c.subject)
        return self.update(slug, group=g, group_by='hand' if by == 'hand' else by if g else None)   # `by`: hand, model, sort (a saved sort) or placed (a new concept)

def _paths(*lists) -> list:
    "The group paths in `lists`, each one time and with its parents, sorted. Case does not count."
    seen = {}
    for g in (g for l in lists for g in l):
        for i in range(1, len(g) + 1): seen.setdefault(tuple(x.lower() for x in g[:i]), list(g[:i]))
    return [seen[k] for k in sorted(seen)]

def _in_use(self:Learning, subject:str='') -> list:
    "The group paths that concepts of `subject` have now."
    return _paths(clean_group(c.get('group'), c.subject) for c in self.concepts(subject or None) if (c.subject or '') == (subject or ''))

def _groups_row(self:Learning, subject:str=''):
    "The document of the group list of `subject`, or None."
    return self.v.by_source(_src('groups', slugify(subject or 'general')))

def _groups_doc(self:Learning, subject:str='') -> dict|None:
    "The meta of the group list of `subject`: `groups` (the kept paths) and `hand` (the paths that you made or renamed); None when there is no list."
    r = _groups_row(self, subject)
    return meta_of(r) if r else None

def _write_groups(self:Learning, subject:str, keep:list, hand:list):
    "Write the group list of `subject`: the kept paths and the hand paths."
    src, m = _src('groups', slugify(subject or 'general')), dict(subject=subject or '', groups=keep, hand=hand)
    r = self.v.put('; '.join(' › '.join(g) for g in keep) or 'no groups', title=f"Groups of {subject or 'general'}", source=src, kind='groups', meta=m)

@patch
def saved_groups(self:Learning, subject:str='') -> list|None:
    "The group paths kept for `subject` when a sort was saved; None when none was."
    m = _groups_doc(self, subject)
    return m.get('groups', []) if m is not None else None

@patch
def hand_groups(self:Learning, subject:str='') -> list:
    "The group paths of `subject` that you made or renamed. The tutor never renames or drops them."
    return (_groups_doc(self, subject) or {}).get('hand', [])

@patch
def save_groups(self:Learning, subject:str='', groups=None, replace:bool=False) -> list:
    "Keep the group paths of `subject`: `groups` (default: the ones in use) added to the ones kept before. A kept group is never dropped by the tutor. With `replace`, the list becomes `groups` and the hand groups. Return the list."
    with self._lock:
        hand, new = self.hand_groups(subject), _in_use(self, subject) if groups is None else groups
        keep = _paths(new, hand) if replace else _paths(self.saved_groups(subject) or [], new, hand)
        _write_groups(self, subject, keep, hand)
        return keep

@patch
def groups(self:Learning, subject:str='') -> list:
    "The group paths of `subject`: the kept ones and the ones in use, each one time and with its parents, sorted."
    return _paths(self.saved_groups(subject) or [], _in_use(self, subject))

@patch
def unsorted(self:Learning, subject:str=None) -> L:
    "The concepts that the tutor may place: no group, and not kept out of the groups by hand."
    return self.concepts(subject).filter(lambda c: not c.get('group') and c.get('group_by') != 'hand')

@patch
def unplace(self:Learning, slug:str) -> AttrDict:
    "Undo a placement by the tutor: no group, and the tutor does not place the concept by itself again. A group that you set, or that a saved sort set, stays."
    with self._lock:
        c = self.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        return self.update(slug, group=[], group_by='undone') if c.get('group_by') == 'placed' else c

@patch
def place(self:Learning, slug:str, path) -> bool:
    "Place a new concept in the group `path`, as the tutor. One step under a lock: it looks at the concept again, and changes nothing when the concept is gone, has a group, was kept out of the groups by hand, or was undone. Return whether it placed the concept."
    with self._lock:
        c = self.concept(slug)
        if c is None or c.get('group') or c.get('group_by') in ('hand', 'undone') or not (g := clean_group(path, c.subject)): return False
        self.update(slug, group=g, group_by='placed')
        return True

#### Sort by hand

You can sort the concepts yourself. `rename` gives a concept a new name and marks it `named_by='hand'`, so a new extract never renames it. The old name stays as an alias, so the find box still finds it.
`rename_group` gives a group a new name. Each concept in the group, or in a group below it, gets the new name and `group_by='hand'`. A name that is the name of a group beside it, in any case and without marks, joins the two groups.
`new_group` adds an empty group. `hand_groups` lists the groups that you made or renamed. "Sort everything again" keeps them.
`snapshot` and `restore` are the Undo of one change by hand: `snapshot` keeps the fields that a change by hand can touch, and `restore` writes them back.

In [ ]:
#| export
def fold(s) -> str:
    "`s` in lower case with no marks (litesearch `fold_text`), so `Maghā` and `MAGHA` are one name, and `संधि` gives `samdhi`."
    return fold_text(str(s or ''))

def _key(g): return tuple(fold(x) for x in g)

def under(g, path) -> bool:
    "Whether the group `g` is the group `path` or a group below it. Case and marks do not count."
    return bool(path) and len(g or []) >= len(path) and _key(g[:len(path)]) == _key(path)

def _uniq(gs) -> list:
    "The paths `gs`, each one time, in their order. Case and marks do not count."
    seen = {}
    for g in gs: seen.setdefault(_key(g), list(g))
    return list(seen.values())

def _spell(gs, g) -> list:
    "`g` with each level spelled as a path in `gs` that is the same name in another case or with marks."
    out = []
    for x in g: out.append(next((p[-1] for p in gs if len(p) == len(out) + 1 and _key(p) == _key([*out, x])), x))
    return out

def _of(self:Learning, subject:str='') -> list:
    "The concepts of `subject` only ('' is the concepts with no subject)."
    return [c for c in self.concepts(subject or None) if (c.subject or '') == (subject or '')]

@patch
def rename(self:Learning, slug:str, name:str) -> AttrDict:
    "Give concept `slug` the name `name`, set by hand: a new extract never renames it. The old name becomes an alias."
    n = clean_name(name, 120)
    if not usable_name(n): raise ValueError('Use two letters or digits or more in a name.')
    with self._lock:
        c = self.concept(slug)
        if c is None: raise KeyError(f'no concept {slug!r}')
        if n == c.name: return self.update(slug, named_by='hand')
        return self.put(n, c.statement, c.subject, slug=slug, aliases=[c.name], named_by='hand')   # `put` makes the search text and the title again

@patch
def rename_group(self:Learning, subject:str, path, name:str) -> list:
    "Give the group `path` of `subject` the last name `name`, set by hand. Its concepts and the groups below it move with it. A name of a group beside it joins the two. Return the slugs changed."
    with self._lock:
        path = _spell(self.groups(subject), clean_group(path, subject))
        new = clean_group([*path[:-1], name], subject)
        if not path or len(new) != len(path): raise ValueError('Use letters or digits in a group name.')
        new = _spell([g for g in self.groups(subject) if _key(g) != _key(path)], new)   # a name of a group beside it: the two groups join
        moved = []
        for c in _of(self, subject):
            if under(g := clean_group(c.get('group'), c.subject), path): self.update(c.slug, group=[*new, *g[len(path):]], group_by='hand'); moved.append(c.slug)
        mv = lambda gs: [[*new, *g[len(path):]] if under(g, path) else list(g) for g in gs]
        _write_groups(self, subject, _paths(mv(self.saved_groups(subject) or []), [new]), _uniq([*mv(self.hand_groups(subject)), new]))
        return moved

@patch
def new_group(self:Learning, subject:str, path) -> list:
    "Add the empty group `path` to `subject`, made by hand. Return the group list."
    with self._lock:
        g = _spell(self.groups(subject), clean_group(path, subject))
        if not g: raise ValueError('Use letters or digits in a group name.')
        _write_groups(self, subject, _paths(self.saved_groups(subject) or [], [g]), _uniq([*self.hand_groups(subject), g]))
        return self.groups(subject)

@patch
def unmark_groups(self:Learning, subject:str, paths) -> list:
    "Take the hand mark off the groups `paths` of `subject`, so the tutor can sort them again. Return the hand groups."
    with self._lock:
        m = _groups_doc(self, subject) or {}
        hand = [g for g in m.get('hand', []) if _key(g) not in {_key(p) for p in paths}]
        _write_groups(self, subject, m.get('groups', []), hand)
        return hand

_HELD = ('name', 'named_by', 'aliases', 'group', 'group_by', 'subject', 'subject_by')

@patch
def snapshot(self:Learning, slugs=(), subjects=()) -> AttrDict:
    "The fields that a change by hand can touch, for the concepts `slugs` and the group lists of `subjects`, so `restore` can write them back."
    return AttrDict(cs={s: {k: c.get(k) for k in _HELD} for s in slugs if (c := self.concept(s))},
                    groups={s: _groups_doc(self, s) for s in subjects})   # None: there was no list

@patch
def restore(self:Learning, snap) -> list:
    "Write back the fields of `snap` (from `snapshot`). A concept that is gone stays gone. Return the slugs written."
    with self._lock:
        for s, m in snap['groups'].items():
            if m is not None: _write_groups(self, s, m.get('groups', []), m.get('hand', []))
            elif (r := _groups_row(self, s)): self.v.forget(r['id'])   # there was no list before: there is none again
        out = []
        for s, m in snap['cs'].items():
            if (c := self.concept(s)) is None: continue
            self.update(s, **{**m, 'name': c.name, 'named_by': None})
            if c.name != m['name']: self.put(m['name'], c.statement, slug=s)   # `put` makes the search text and the title again
            self.update(s, named_by=m['named_by']); out.append(s)
        return out

`set_subject` gives a subject to a concept. `readable_name` and `repair_names` make a readable name for a concept whose name is its slug.

In [ ]:
#| export
def clean_subject(s) -> str:
    "`s` as one short plain lower-case name; '' when it has none."
    return ' '.join(_plain(s).split())[:GROUP_LEN].strip(' -').lower() if isinstance(s, str) else ''

@patch
def set_subject(self:Learning, slug:str, subject:str, by:str='model') -> AttrDict:
    "Give concept `slug` a subject, set `by` the learner (`hand`) or the tutor (`model`). The tutor sets only an empty subject, and never on a concept that you moved by hand."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    s = clean_subject(subject)
    if by != 'hand' and (c.get('group_by') == 'hand' or c.subject or not s): return c
    return self.put(c.name, c.statement, s, slug=slug, subject_by=by)   # `put` makes the search text and the title again

def readable_name(name, statement:str='') -> str:
    "`name` as words: a name that is a slug with hyphens, or an empty name (words from `statement`), becomes a short sentence-case name. Else `name`."
    n = ' '.join(str(name or '').split())
    if n and not (' ' not in n and '-' in n and n == slugify(n)): return n
    n = ' '.join((n or ' '.join(str(statement or '').split()[:6])).replace('-', ' ').split()).rstrip('.,;:')
    return n[:1].upper() + n[1:]

@patch
def repair_names(self:Learning) -> list:
    "One time for each shelf, give each concept whose name is its own slug, with hyphens, a readable name. Return the slugs that changed. A name that differs from the slug, or that you set by hand (`named_by`), stays."
    if self.v.by_source(_src('marker', 'names')): return []
    out = [c.slug for c in self.concepts() if c.name == c.slug and '-' in c.name and c.get('named_by') != 'hand']   # `named_by='hand'`: no rename path yet, the editors of 0.0.13 set it
    for s in out: c = self.concept(s); self.put(readable_name(s), c.statement, c.subject, slug=s)   # `put` makes the search text and the title again
    self.v.add('The names of the concepts were repaired.', title='Names repaired', source=_src('marker', 'names'), kind='marker')
    return out

`tree` puts concepts in a tree for the Concepts page: a node for each subject, then its groups, then their groups. A node has its `name`, its `path` under the subject, `n` (the concepts in it and below it), `kids` (its groups by name) and `cs` (its own concepts). A concept with no group is in `cs` of its subject. Two names that differ only in case are one group.

In [ ]:
#| export
def _node(name, path): return AttrDict(name=name, path=path, kids={}, cs=[])

def tree(concepts, groups=None) -> list:
    "The subject nodes of `concepts`, sorted by name, each with its groups and the count of concepts below it. `groups` (`{subject: paths}`) adds groups with no concept, under a subject that shows."
    root = {}
    def walk(n, g):
        for x in g: n = n.kids.setdefault(x.lower(), _node(x, [*n.path, x]))
        return n
    for c in concepts:
        s = c.get('subject') or ''
        walk(root.setdefault(s.lower(), _node(s, [])), clean_group(c.get('group'), s)).cs.append(c)
    for s, gs in (groups or {}).items():
        if (s or '').lower() in root:
            for g in gs: walk(root[(s or '').lower()], g)
    def done(n):
        n.kids = [done(k) for _, k in sorted(n.kids.items())]
        n.cs = sorted(n.cs, key=lambda c: (c.get('order', 1e9), c['name'].lower()))
        n.n = len(n.cs) + sum(k.n for k in n.kids)
        return n
    return [done(n) for _, n in sorted(root.items())]

### Attempts

Each answer is one attempt document. Thus search can find what the learner said. The tutor can also find old questions and not ask them again.

In [ ]:
#| export
def attempt_text(a) -> str:
    "The searchable text of an attempt: the question, the answer, the think-aloud and the feedback."
    parts = [f"Question: {a['question']}", a.get('answer') and f"Answer: {a['answer']}",
             a.get('thinking') and f"Thinking: {a['thinking']}", a.get('feedback') and f"Feedback: {a['feedback']}"]
    return '\n\n'.join(p for p in parts if p)

`record` saves one attempt. The vault joins two sources with the same text into one document. So `record` asks for a new document each time (`force=True`): two attempts with the same words stay two records. Each one has its own source, `learning://attempt/<id>`.

In [ ]:
#| export
@patch
def record(self:Learning, concept:str, question:str, answer:str='', thinking:str='', correct:bool=False,
           confidence:int=None, depth:int=1, kind:str='explain', hints:int=0, dont_know:bool=False, grade:int=None,
           feedback:str='', first_error:str='', misconception:str='', session:str='', **meta) -> AttrDict:
    "File one attempt as a vault document, so the learner can search what they said."
    a = AttrDict(id=uuid.uuid4().hex[:12], concept=concept, question=question, answer=answer, thinking=thinking,
                 correct=correct, confidence=confidence, depth=depth, kind=kind, hints=hints, dont_know=dont_know,
                 grade=grade, feedback=feedback, first_error=first_error, misconception=misconception,
                 session=session, at=time.time(), **meta)
    c = self.concept(concept)
    self.v.add(attempt_text(a), title=f"{c.name if c else concept}: {question[:80]}", source=_src('attempt', a.id),
               kind='attempt', meta=dict(a), force=True)   # its own document, also when an older attempt has the same text
    return a

`attempts` reads them back, with the newest first.

In [ ]:
#| export
@patch
def attempts(self:Learning, concept:str=None, session:str=None, limit:int=50, since:float=None) -> L:
    "Recent attempts, newest first, for one concept or one session; with `since`, only the attempts from that time on."
    w, args = [], []
    if since is not None: w.append("json_extract(meta, '$.at')>=?"); args.append(since)
    if concept: w.append("json_extract(meta, '$.concept')=?"); args.append(concept)
    if session: w.append("json_extract(meta, '$.session')=?"); args.append(session)
    rs = L(self._docs('attempt', ' and '.join(w), args)).map(lambda r: AttrDict(meta_of(r)))
    return L(sorted(rs, key=lambda a: -a.at)[:limit]).map(lambda a: AttrDict({k: v for k, v in a.items() if k not in _SKIP}))

`mark_attempt` adds fields to a saved attempt, for example a lesson that you asked for later. `add_gap` keeps a gap that you named on a concept.

In [ ]:
#| export
@patch
def mark_attempt(self:Learning, aid:str, **kv) -> dict:
    "Merge `kv` into the meta of attempt `aid`."
    r = self.v.by_source(_src('attempt', aid))
    if r is None: raise KeyError(f'no attempt {aid!r}')
    return self.v.set_meta(r['id'], **kv)

@patch
def add_gap(self:Learning, slug:str, gap:str):
    "Record that you said you do not know concept `gap` while you learned `slug`."
    c = self.concept(slug)
    if c is None: raise KeyError(f'no concept {slug!r}')
    if gap != slug and gap not in (c.get('gaps') or []): self.update(slug, gaps=[*(c.get('gaps') or []), gap])

### Sessions

A session summary is one document. A new summary with the same id replaces the old one.

In [ ]:
#| export
@patch
def log_session(self:Learning, sid:str, summary:str, **meta):
    "File a session, and replace an earlier one with the same id. The meta changes also when the summary text does not."
    m = dict(id=sid, summary=summary, **meta)
    r = self.v.put(summary, title=f'Session {sid}', source=_src('session', sid), kind='session', meta=m)
    return r

@patch
def session(self:Learning, sid:str) -> AttrDict|None:
    "One filed session by id, as its meta."
    r = self.v.by_source(_src('session', sid))
    return AttrDict({**meta_of(r), 'doc_id': r['id']}) if r else None

@patch
def sessions(self:Learning, limit:int=100, journey:str=None) -> L:
    "The filed sessions, the most recently changed first; with `journey`, only the sessions of that walk."
    w, args = ("json_extract(meta, '$.journey')=?", [journey]) if journey else ('', [])
    rs = L(self._docs('session', w, args)).map(lambda r: AttrDict({**meta_of(r), 'doc_id': r['id']}))
    return L(sorted(rs, key=lambda s: -(s.get('updated') or s.get('end') or s.get('start') or 0))[:limit])

### Search and graph

`search` uses the hybrid search of the vault. Each hit has the kind, title and meta of its document. The method stats, the marker of a repair, a session with no answers and an open question stay out of it. Two attempts with the same text show as one hit. `graph` gives the nodes and links for the graph view.

In [ ]:
#| export
def _answers(m) -> int:
    "How many answers a filed session has: its log, or `asked` for an old one with no log."
    return len(m['log']) if 'log' in m else m.get('asked', 0)

NO_TEXT = ('stats', 'marker', 'groups', 'open-q', 'edits', 'jobs')   # the kinds that the search leaves out
NOT_FOUND = ','.join(f'-{k}' for k in NO_TEXT)   # the kind filter of vishalakshi that leaves them out

@patch
def search(self:Learning, q:str, kind:str=None, limit:int=10) -> L:
    """Hybrid search over the shelf. Each hit also has the `kind`, `title` and `meta` of its document, so a page can show it
    by kind. The method stats, the marker of a repair, the history of the changes to a talk or a walk, a session with no answers and an open question stay out: they have no text that you wrote or learned,
    and an open question has its answer key."""
    n = limit + 10
    while True:   # fetch more until there are `limit` real hits, or the search has no more
        hs = L(self.v.search(q, limit=n, kind=kind or NOT_FOUND))
        ids = list(dict.fromkeys(h['doc_id'] for h in hs))
        rows = {r['id']: r for r in self.v.t.docs(where=f"id in ({','.join('?' * len(ids))})", where_args=ids)} if ids else {}
        out, seen = L(), set()
        for h in hs:
            if (r := rows.get(h['doc_id'])) is None: continue
            m = meta_of(r)
            if r['kind'] in NO_TEXT or (r['kind'] == 'session' and not _answers(m)): continue
            if r['kind'] == 'attempt':   # two attempts with the same words are two records, but one hit
                if (k := (r['title'], h.get('snippet'))) in seen: continue
                seen.add(k)
            out.append({**h, 'kind': r['kind'], 'title': r['title'], 'meta': m})
        if len(out) >= limit or len(hs) < n: return out[:limit]
        n *= 4

@patch
def graph(self:Learning, subject:str=None) -> dict:
    "Nodes and links (prereq, related, gap) for a graph view. A link to a slug that has no concept is dropped."
    cs = self.concepts(subject).filter(listed)   # a concept that an older tool filed as `x` stays out of the view
    have = set(cs.attrgot('slug'))
    nodes = [dict(id=c.slug, name=c.name, subject=c.subject, floor=c.floor, ceiling=c.ceiling, due=c.due, seen=c.seen, group=clean_group(c.get('group'), c.subject)) for c in cs]
    links = [dict(source=p, target=c.slug, kind='prereq') for c in cs for p in c.prereqs if p in have]
    links += [dict(source=c.slug, target=r, kind='related') for c in cs for r in c.related if r in have and c.slug < r]
    links += [dict(source=g, target=c.slug, kind='gap') for c in cs for g in c.get('gaps') or () if g in have]
    return dict(nodes=nodes, links=links)

### A shelf for tests

Each test uses a vault in a temporary folder with the offline hashing encoder.
The tests never open the real vault of the learner.

In [ ]:
#| export
def temp_learning(**kw) -> Learning:
    "A `Learning` on a new vault in a temporary folder (vishalakshi `Vault.temp`), with the offline hashing encoder."
    return Learning(Vault.temp(), **kw)

### Tests

`put` makes a concept with the start state and the slug from its name.

In [ ]:
lr = temp_learning()
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank) for each item.', 'ml', prereqs=['ranking'], aliases=['RRF'])
test_eq(c.slug, 'reciprocal-rank-fusion')
test_eq(c.floor, 0)
test_eq(c.prereqs, ['ranking'])

`update` changes the state.

In [ ]:
c = lr.update(c.slug, floor=3, stability=2.)
test_eq((c.floor, c.stability), (3, 2.))

A concept keeps its `core` and its wrong ideas. A later `put` with no `core` keeps them. A wrong idea as a plain string becomes a dict.

In [ ]:
wl = temp_learning()
c = wl.put('Token embeddings', 'Each token ID picks a row of a learned matrix.', 'ml', core='Each token gets a learned vector, so tokens used in the same way end up close.',
           wrong=['the ID number itself carries meaning', dict(think='the vectors are fixed', fact='training changes them')])
test_eq(c.wrong, [dict(think='the ID number itself carries meaning', fact=''), dict(think='the vectors are fixed', fact='training changes them')])
test_eq(wl.put('Token embeddings', 'Rows of E.').core, c.core)

A second `put` keeps the learner state and merges the lists, also when the text changes.

In [ ]:
c = lr.put('Reciprocal rank fusion', 'Merge ranked lists: score is the sum of 1/(k + rank).', related=['bm25'], aliases=['RRF', 'rank fusion'])
test_eq(c.floor, 3)
test_eq(c.subject, 'ml')
test_eq(c.aliases, ['RRF', 'rank fusion'])
test_eq(c.related, ['bm25'])
test_eq(len(lr.concepts()), 1)

`concepts` and `subjects` filter by subject. `graph` keeps only the links between known concepts.

In [ ]:
lr.put('Ranking', 'Order items by a score.', 'ml')
lr.put('Sandhi', 'Sounds change where words meet.', 'sanskrit')
test_eq(len(lr.concepts('ml')), 2)
test_eq({r['subject'] for r in lr.subjects()}, {'ml', 'sanskrit'})
g = lr.graph()
test_eq(g['links'], [dict(source='ranking', target='reciprocal-rank-fusion', kind='prereq')])

`record` saves an attempt. `attempts` and `search` find it again.

In [ ]:
a = lr.record('reciprocal-rank-fusion', 'Why does RRF ignore raw scores?', answer='Because scores from different systems are not comparable.',
              thinking='bm25 and cosine have different scales', correct=True, confidence=80, depth=3)
test_eq(lr.attempts('reciprocal-rank-fusion')[0].id, a.id)
test_eq(lr.attempts('ranking'), [])
test(lr.search('scales comparable', kind='attempt'), [], lambda a,b: len(a) > 0)

`drop` removes a concept.

In [ ]:
lr.drop('ranking')
test_is(lr.concept('ranking'), None)

#### Shelves of documents

A real vault also keeps a store of names, `entities`, with no `doc_id` column. `shelf_names` leaves it out. It keeps the main shelf, the learning shelf and a shelf of notebooks.

In [ ]:
dv = temp_learning()
dv.vault.shelf('notebooks').add('Notes on softmax.', title='nbs/softmax', source='/x/softmax.md', kind='md')
dv.vault.db.get_graph('store', ann=False)   # the store of names, as vishalakshi makes it
assert 'entities' in dv.vault.db.table_names()   # the store of names is in the vault, though vishalakshi 0.1.36 no longer lists it as a shelf
test_eq(sorted(dv.vault.shelf_names()), ['learning', 'notebooks', 'store'])

`shelf_names` reads only the names of the stores. It does not count the documents of each store, so SQLite logs no error for a store of names with no `docs` table. A page can call it many times.

In [ ]:
import logging

In [ ]:
class _Seen(logging.Handler):
    def __init__(self): super().__init__(); self.got = []
    def emit(self, r): self.got.append(r.getMessage())
seen, root = _Seen(), logging.getLogger()
root.addHandler(seen)
try: test_eq(sorted(dv.vault.shelf_names()), ['learning', 'notebooks', 'store'])
finally: root.removeHandler(seen)
test_eq([m for m in seen.got if 'no such table' in m], [])

`doc_shelf` opens a shelf of documents. It gives None for the store of names and for a name that is not in the vault, and it does not make that shelf.

In [ ]:
test_eq(dv.doc_shelf('notebooks').t.docs()[0]['title'], 'nbs/softmax')
test_is(dv.doc_shelf('store'), dv.vault)
test_is(dv.doc_shelf('entities'), None)
test_is(dv.doc_shelf('nope'), None)
assert 'nope' not in [s['store'] for s in dv.vault.shelves()]

The search fetches more hits until it has `limit` real hits, or the shelf has no more. Here 30 sessions with no answers match the words best. Three concepts match only one word. All three concepts come back.

In [ ]:
sl = temp_learning()
for k in range(30): sl.log_session(f'empty{k}', f'zebra quagga okapi zebra quagga okapi {k}', start=1., log=[])
for k in range(3): sl.put(f'Giraffe {k}', f'A tall animal {k}, once next to an okapi.', 'zoo')
test_eq(len(sl.search('zebra quagga okapi', limit=3)), 3)

In [ ]:
lr.log_session('s1', '1 of 2 passed.', start=1., updated=2., asked=2)
lr.log_session('s1', '1 of 2 passed.', start=1., updated=3., asked=2, ended=3.)   # the same text: the meta still changes
test_eq((lr.session('s1').updated, lr.session('s1').ended, lr.session('s1').summary), (3., 3., '1 of 2 passed.'))
test_eq(len(lr.sessions()), 1)

`sessions` lists the most recently changed session first.

In [ ]:
lr.log_session('s2', 'Started.', start=5., updated=5.)
test_eq(lr.sessions().attrgot('id'), ['s2', 's1'])
test_is(lr.session('nope'), None)

With `journey`, `sessions` reads only the sessions of that walk, in the store.

In [ ]:
lr.log_session('s3', 'A walk.', start=6., journey='j1')
test_eq(lr.sessions(journey='j1').attrgot('id'), ['s3'])
test_eq(len(lr.sessions()), 3)

`search` leaves out a session with no answers and the method stats. The session is still filed, so you can continue it. Each hit names its kind.

In [ ]:
lr.log_session('s4', '0 of 0 passed. Concepts: none.', start=7., log=[])
lr.v.add('How you learn: method counts, passed.', title='How you learn', source='stats:test', kind='stats')
hs = lr.search('passed', limit=20)
test_eq([h for h in hs if h['kind'] == 'stats' or h['meta'].get('id') == 's4'], [])
test_eq('s1' in [h['meta'].get('id') for h in hs if h['kind'] == 'session'], True)   # a session with answers stays
test_eq(lr.session('s4').id, 's4')

The search fetches more hits until it has `limit` real hits, or the shelf has no more. Here 30 sessions with no answers match the words first, and three concepts match them too. All three concepts come back.

`concept_by_doc` gives the concept of a search hit. It gives None for a hit that is not a concept.

In [ ]:
cl = temp_learning()
cl.put('Reciprocal rank fusion', 'Merge ranked lists by summing 1/(k + rank).', 'ml')
cl.record('reciprocal-rank-fusion', 'Why ranks?', answer='scales differ')
test_eq(cl.concept_by_doc(cl.search('rank fusion', kind='concept')[0]['doc_id']).slug, 'reciprocal-rank-fusion')
test_is(cl.concept_by_doc(cl.search('scales differ', kind='attempt')[0]['doc_id']), None)

`slugify` reads Devanagari and Telugu, keeps C, C++ and C# apart, and gives the same hash slug for the same unreadable name.

In [ ]:
test_eq([slugify(s) for s in ('saṃdhi', 'संधि', 'గుణ', 'C', 'C++', 'C#', 'R&D')], ['samdhi', 'samdhi', 'guna', 'c', 'c-plus-plus', 'c-sharp', 'r-and-d'])
test_eq(slugify('学习'), slugify('学习'))
assert slugify('学习').startswith('c-')

This test makes a vault for `reslug` as abhyasa 0.0.5 left it: C# saved as `c`, a Devanagari name with a random slug, and links, an attempt and a session that use these slugs.

In [ ]:
ol = temp_learning()
ol.put('C#', 'A language from Microsoft.', 'code', slug='c')
ol.put('संधि', 'Sounds change where words meet.', 'sanskrit', slug='a1b2c3d4')
ol.put('karaka term', 'A set slug.', 'sanskrit', slug='sa-term-karaka')
ol.put('Ranking', 'Order items by a score.', 'ml', prereqs=['c'], related=['a1b2c3d4', 'sa-term-karaka'])
ol.record('c', 'What is C#?', answer='c')
ol.log_session('s1', 'One question.', concepts=['c'], log=[dict(concept='c', question='What is C#?')], focus='a1b2c3d4')

A dry run lists the moves and changes nothing. A slug that a source sets is not in the list.

In [ ]:
test_eq(ol.reslug(dry_run=True), {'c': 'c-sharp', 'a1b2c3d4': 'samdhi'})
assert ol.concept('c') is not None and ol.concept('c-sharp') is None

The real run moves the concepts and every slug that points to them. An answer that is the text `c` stays as it is. A second run has nothing to do.

In [ ]:
test_eq(ol.reslug(), {'c': 'c-sharp', 'a1b2c3d4': 'samdhi'})
test_eq((ol.concept('c'), ol.concept('a1b2c3d4')), (None, None))
test_eq((ol.concept('c-sharp').name, ol.concept('samdhi').statement), ('C#', 'Sounds change where words meet.'))
r = ol.concept('ranking')
test_eq((r.prereqs, r.related), (['c-sharp'], ['samdhi', 'sa-term-karaka']))
a = ol.attempts('c-sharp')[0]
test_eq((a.concept, a.answer), ('c-sharp', 'c'))
s = ol.session('s1')
test_eq((s.concepts, s.log[0]['concept'], s.focus), (['c-sharp'], 'c-sharp', 'samdhi'))
test_eq(ol.reslug(), {})

`mark_attempt` changes a saved attempt. `add_gap` keeps each gap one time, and `graph` shows it as a `gap` link.

In [ ]:
gl = temp_learning()
gl.put('Functions and the neuron', 'z = w . x + b.', 'ml'); gl.put('Weighted sum', 'Add each input times its weight.', 'ml')
a = gl.record('functions-and-the-neuron', 'Why does z double?', answer='2z')
test_eq(gl.mark_attempt(a.id, taught=True)['taught'], True)
test_eq(gl.attempts('functions-and-the-neuron')[0].taught, True)
gl.add_gap('functions-and-the-neuron', 'weighted-sum'); gl.add_gap('functions-and-the-neuron', 'weighted-sum')
test_eq(gl.concept('functions-and-the-neuron').gaps, ['weighted-sum'])
assert dict(source='weighted-sum', target='functions-and-the-neuron', kind='gap') in gl.graph()['links']

Two attempts with the same text are two records. The vault does not join them, so each one counts in your history and in the misses that the tutor counts. A field added later goes on the right attempt.

In [ ]:
ld = temp_learning(); ld.put('Ranking', 'Order items by a score.', 'ml')
a1, a2 = [ld.record('ranking', 'What is a rank?', 'a score', feedback='Look at positions.', final=True) for _ in range(2)]
test_eq(ld.attempts('ranking').attrgot('id'), [a2.id, a1.id])
ld.mark_attempt(a2.id, taught=1)
test_eq([a.get('taught') for a in ld.attempts('ranking')], [1, None])

A search shows the same attempt text only once.

In [ ]:
test_eq(len([h for h in ld.search('What is a rank', kind='attempt') if h['kind'] == 'attempt']), 1)

### Groups

An old vault has no groups. A concept loads with an empty path, and no one set it.

In [ ]:
gv = temp_learning()
m = gv.put('Magha', 'The tenth nakshatra.', 'jyotisha')
test_eq((m.get('group'), m.get('group_by')), (None, None))

`clean_group` keeps at most two short plain names. It drops marks, empty parts, a repeat and a first part that is the subject. A text splits at `/`, `>` or `›`.

In [ ]:
test_eq(clean_group(['Nakshatras', 'Padas', 'More']), ['Nakshatras', 'Padas'])
test_eq(clean_group(' nakshatras  /  padas! '), ['nakshatras', 'padas'])
test_eq(clean_group(['Nakshatras!', '  deva   gana? ']), ['Nakshatras', 'deva gana'])
test_eq(clean_group(['jyotisha', 'nakshatras'], 'jyotisha'), ['nakshatras'])
test_eq(clean_group(['', 'Nakshatras', 'nakshatras', 3]), ['Nakshatras'])
test_eq(clean_group('saṃdhi › स्वर'), ['saṃdhi', 'स्वर'])
test_eq(len(clean_group(['x' * 90])[0]), GROUP_LEN)
test_eq((clean_group(None), clean_group({'a': 1}), clean_group('')), ([], [], []))

`set_group` by the model moves a concept. A move by hand sets `group_by='hand'`. After that, a move by the model changes nothing.

In [ ]:
test_eq(gv.set_group('magha', ['nakshatras'], by='model').group, ['nakshatras'])
m = gv.set_group('magha', 'Nakshatras / Gana')
test_eq((m.group, m.group_by), (['Nakshatras', 'Gana'], 'hand'))
test_eq(gv.set_group('magha', ['rasis'], by='model').group, ['Nakshatras', 'Gana'])

A new `put` with a group, as from a new source, does not move a concept that you moved by hand. It keeps the learner state.

In [ ]:
m = gv.put('Magha', 'The tenth nakshatra, ruled by Ketu.', 'jyotisha', group=['rasis'], group_by='model')
test_eq((m.group, m.group_by, m.statement), (['Nakshatras', 'Gana'], 'hand', 'The tenth nakshatra, ruled by Ketu.'))

An empty path by hand takes the concept out of its group and keeps the mark. So the tutor does not sort it again, and a move by the model changes nothing.

In [ ]:
m = gv.set_group('magha', '')
test_eq((m.group, m.group_by), ([], 'hand'))
test_eq(gv.set_group('magha', ['rasis'], by='model').group, [])

`groups` lists the paths in use under one subject, with their parents. A path in another subject does not show.

`set_subject` gives a subject to a concept that has none. A move by the tutor never changes a concept that you moved by hand, and it never replaces a subject. A move by hand always works.

In [ ]:
sj = temp_learning()
sj.put('Alpha', 'One.'); sj.put('Beta', 'Two.'); sj.put('Gamma', 'Three.', 'ml')
sj.set_group('beta', 'mine')
test_eq(sj.set_subject('alpha', ' Star  Maps! ').subject, 'star maps')
test_eq(sj.concept('alpha').subject_by, 'model')
test_eq(sj.set_subject('beta', 'stars').subject, '')                 # moved by hand: the tutor does not touch it
test_eq(sj.set_subject('gamma', 'stars').subject, 'ml')              # it has a subject
test_eq(sj.set_subject('gamma', 'stars', by='hand').subject, 'stars')

A name that is its own slug, with hyphens, gets a readable name one time. A name that differs from the slug stays. The empty name of the finder gets words from the statement.

In [ ]:
test_eq(readable_name('attention-heads-and-anaphora-resolution'), 'Attention heads and anaphora resolution')
test_eq(readable_name('Softmax'), 'Softmax'); test_eq(readable_name('softmax'), 'softmax'); test_eq(readable_name('my-Note'), 'my-Note')
test_eq(readable_name('', 'A rank is a position in a list. More.'), 'A rank is a position in')
rn = temp_learning()
rn.put('attention-heads-and-anaphora-resolution', 'Heads.', 'ml', slug='attention-heads-and-anaphora-resolution')
rn.put('my-note', 'Mine.', 'ml', slug='note')
rn.put('foo-bar', 'Typed.', 'ml', slug='foo-bar', named_by='hand')
def old_vault(l):
    "Forget the mark of the repair, as in a vault from before it."
    l.v.forget(first(l.v.t.docs(where='source=?', where_args=['learning://marker/names']))['id'])
old_vault(rn)
test_eq(Learning(rn.vault).concept('attention-heads-and-anaphora-resolution').name, 'Attention heads and anaphora resolution')
test_eq((rn.concept('note').name, rn.concept('foo-bar').name), ('my-note', 'foo-bar'))   # a name that differs from the slug, or that you typed, stays
test_eq(rn.repair_names(), [])

The repair runs one time: a mark on the shelf says that it ran. A vault that cannot be read or changed does not break the load. A change of name or subject also makes the search text and the title again.

In [ ]:
from unittest.mock import patch as mock_patch

In [ ]:
rn.put('late-slug-name', 'Late.', 'ml', slug='late-slug-name')
test_eq(rn.repair_names(), [])                                        # the mark is set: it does not look again
test_eq(rn.concept('late-slug-name').name, 'late-slug-name')
old_vault(rn)
with mock_patch.object(Learning, 'repair_names', side_effect=OSError('read only')): Learning(rn.vault)   # the load goes on
test_eq(rn.repair_names(), ['late-slug-name'])
doc = lambda l, s: first(l.v.t.docs(where='source=?', where_args=[f'learning://concept/{s}']))
test_eq(doc(rn, 'late-slug-name')['title'], 'Late slug name')
sj.set_subject('gamma', 'cosmos', by='hand')
test_eq(sj.concept('gamma').subject, 'cosmos')
assert sj.concept('gamma').doc_id in [h['doc_id'] for h in sj.v.search('cosmos', 3)]   # the search text has the new subject

The mark of the repair is a document, but search never gives it.

In [ ]:
mt = temp_learning(); mt.put('Names', 'Names of concepts repaired.', 'ml')
assert first(mt.v.t.docs(where='kind=?', where_args=['marker']))
test_eq([h['kind'] for h in mt.search('names concepts repaired')], ['concept'])
test_eq(mt.search('Names repaired', kind='marker'), [])

#### Saved groups stay

When you save a sort, the tutor keeps the group list of each subject and does not move a sorted concept again. `save_groups` keeps the list. `groups` gives the saved groups and the groups in use. A saved group stays when no concept is in it. `unsorted` lists the concepts that the tutor may still place: no group, and not set by hand.

In [ ]:
cl = temp_learning()
for n in ('Ashvini', 'Bharani', 'Krittika'): cl.put(n, f'{n}.', 'jyotisha')
test_is(cl.saved_groups('jyotisha'), None)
cl.set_group('ashvini', ['Stars'], by='sort')
test_eq(cl.save_groups('jyotisha'), [['Stars']])
cl.set_group('ashvini', [], by='hand'); cl.set_group('bharani', ['Signs'], by='sort')
test_eq(cl.save_groups('jyotisha'), [['Signs'], ['Stars']])           # a kept group stays when no concept is in it
test_eq(cl.groups('jyotisha'), [['Signs'], ['Stars']])
test_eq(cl.saved_groups('JYOTISHA'), [['Signs'], ['Stars']])
test_eq(cl.concept('bharani').group_by, 'sort')

A concept with a group is sorted. A concept with no group is not sorted, except when you set it by hand to no group. `unplace` is the Undo of a placement by the tutor: the concept has no group, and the tutor does not place it by itself again.

In [ ]:
test_eq(cl.unsorted('jyotisha').attrgot('slug'), ['krittika'])         # ashvini: kept out by hand; bharani: sorted
cl.set_group('krittika', ['Stars'], by='placed')
test_eq(cl.unplace('krittika').group_by, 'undone')
test_eq((cl.concept('krittika').group, cl.unsorted('jyotisha').attrgot('slug')), ([], ['krittika']))
test_eq(cl.unplace('bharani').group, ['Signs'])                         # only a placement by the tutor can be undone
test_eq(cl.unplace('ashvini').group_by, 'hand')

A name with fewer than two characters, or with no letter or digit, is not an idea: `usable_name` says so, `idea_name` is the rule for the name of a new concept: a letter or a digit, and not one small letter such as `x`. One capital letter, such as `C`, is a name. `put` refuses a new concept with another name, so no tool, plan or text can file one. A plan or a gap has no source text, so it also needs `real_idea`: a statement of 3 words or more. Library and the graph leave out a concept that breaks the name rule or has a statement of one word (`listed`), such as the `x` that an older tool filed. The marker and the group list never show in search.

In [ ]:
assert usable_name('Ab') and usable_name('x2') and usable_name('C#') and not usable_name('x') and not usable_name(' - ') and not usable_name(None)
test_fail(lambda: cl.put('x', 'x', 'code'), contains='not the name of an idea')         # one small letter
test_fail(lambda: cl.put('z', 'z is a letter for a value.', 'code'), contains='not the name of an idea')
test_eq(cl.put('C', 'C is a compiled systems language.', 'code').name, 'C')            # one capital letter
assert cl.concept('x') is None and listed(cl.concept('c')) and not listed(dict(name='Bias', statement='x'))
test_eq((real_idea('Bias', 'x'), real_idea('C', 'C is a language.')), (False, True))   # with no source text, a statement of 3 words or more
cl.drop('c')
test_eq(cl.search('Groups of jyotisha', kind='groups'), [])
assert not [h for h in cl.search('Groups of jyotisha') if h['kind'] == 'groups']

In [ ]:
gv.put('Ashvini', 'The first nakshatra.', 'jyotisha', group=['nakshatras', 'deva gana'])
gv.put('Mesha', 'The first rasi.', 'jyotisha', group=['Rasis'])
gv.put('Sandhi', 'Sounds change where words meet.', 'sanskrit', group=['phonology'])
test_eq(gv.groups('jyotisha'), [['nakshatras'], ['nakshatras', 'deva gana'], ['Rasis']])
test_eq(gv.groups('ml'), [])

`tree` puts the concepts under their subject, then their groups. Each node has the count of the concepts in it and below it. A concept with no group is an item of its subject. A group name in another case is the same group.

In [ ]:
gv.put('Bharani', 'The second nakshatra.', 'jyotisha', group=['Nakshatras'])
t = tree(gv.concepts())
test_eq([(s.name, s.n) for s in t], [('jyotisha', 4), ('sanskrit', 1)])
j = t[0]
test_eq([(k.name, k.n) for k in j.kids], [('nakshatras', 2), ('Rasis', 1)])
test_eq([c.name for c in j.cs], ['Magha'])
test_eq([(k.path, [c.name for c in k.cs]) for k in j.kids[0].kids], [(['nakshatras', 'deva gana'], ['Ashvini'])])
test_eq([c.name for c in j.kids[0].cs], ['Bharani'])

#### Sort by hand

A renamed group takes its concepts with it. They and the group get the hand mark, so the tutor never moves them.

In [ ]:
hs = temp_learning()
for n in ('Ashvini', 'Bharani'): hs.put(n, f'{n}.', 'jyotisha', group=['nakshatras'])
hs.save_groups('jyotisha')
test_eq(sorted(hs.rename_group('jyotisha', ['Nakshatras'], 'Lunar mansions')), ['ashvini', 'bharani'])
test_eq({(tuple(c.group), c.group_by) for c in hs.concepts('jyotisha')}, {(('Lunar mansions',), 'hand')})
test_eq((hs.hand_groups('jyotisha'), hs.groups('jyotisha')), ([['Lunar mansions']], [['Lunar mansions']]))
assert not hs.unsorted('jyotisha') and not hs.place('ashvini', ['Other'])

A name that is the name of a group beside it, in another case, joins the two groups. The group below a renamed group moves with it.

In [ ]:
hs.put('Pada', 'A quarter.', 'jyotisha', group=['Stars', 'padas']); hs.put('Pada two', 'A quarter.', 'jyotisha', group=['Stars x'])
hs.rename_group('jyotisha', ['Stars x'], 'STARS')
test_eq(hs.concept('pada-two').group, ['Stars'])
hs.rename_group('jyotisha', ['Stars'], 'Asterisms')
test_eq((hs.concept('pada').group, hs.concept('pada-two').group), (['Asterisms', 'padas'], ['Asterisms']))
assert ['Stars'] not in hs.groups('jyotisha') and ['Stars x'] not in hs.groups('jyotisha')
test_fail(lambda: hs.rename_group('jyotisha', ['Asterisms'], '!!!'), contains='letters or digits')

`new_group` adds an empty group that stays. `save_groups(replace=True)` makes the list again and keeps the hand groups. `unmark_groups` takes the mark off.

In [ ]:
hs.new_group('jyotisha', 'Rasis')
assert ['Rasis'] in hs.groups('jyotisha') and ['Rasis'] in hs.hand_groups('jyotisha')
test_eq(hs.save_groups('jyotisha', [['Grahas']], replace=True), [['Asterisms'], ['Grahas'], ['Lunar mansions'], ['Rasis']])
assert ['Asterisms', 'padas'] in hs.groups('jyotisha')   # a group in use still shows
hs.unmark_groups('jyotisha', [['rasis']])
assert ['Rasis'] not in hs.hand_groups('jyotisha') and ['Rasis'] in hs.groups('jyotisha')

A name that you set stays when a new extract puts the concept again. The old name becomes an alias.

In [ ]:
hs.put('Self-attention', 'x', 'ml'); c = hs.rename('self-attention', 'Attention over one sequence')
test_eq((c.name, c.named_by, c.aliases), ('Attention over one sequence', 'hand', ['Self-attention']))
test_eq(hs.put('Self-attention', 'y', 'ml').name, 'Attention over one sequence')
test_fail(lambda: hs.rename('self-attention', ' x '), contains='two letters')
test_eq(hs.rename('self-attention', 'Self\x00 at\x1b[31mtention\n' + 'x' * 200).name, 'Self at [31mtention ' + 'x' * 100)   # no control characters, and a cap
hs.rename('self-attention', 'Attention over one sequence')
test_eq(hs.put('Bad\x07 name', 'x', 'ml').name, 'Bad name')

`snapshot` and `restore` undo one change by hand: the groups of the concepts, their names and the group list.

In [ ]:
snap = hs.snapshot(['pada', 'self-attention'], ['jyotisha'])
hs.rename_group('jyotisha', ['Asterisms'], 'Stars again'); hs.rename('self-attention', 'Self focus'); hs.new_group('jyotisha', 'Extra')
test_eq(sorted(hs.restore(snap)), ['pada', 'self-attention'])
test_eq((hs.concept('pada').group, hs.concept('self-attention').name), (['Asterisms', 'padas'], 'Attention over one sequence'))
assert ['Extra'] not in hs.groups('jyotisha') and ['Asterisms'] in hs.saved_groups('jyotisha')
snap = hs.snapshot([], ['ml']); test_is(hs.saved_groups('ml'), None)
hs.new_group('ml', 'Heads'); hs.restore(snap)
test_is(hs.saved_groups('ml'), None)   # Undo takes away a list that was not there

## The learner model

The learner model: how deep you know each concept, when to review it, and what to ask next.

This module holds the learner model as plain functions with no model calls and no storage.
It finds what an answer means, updates the state of a concept, finds the next depth and picks the next concept.
The tutor calls these functions and the store keeps their results.

Import the tools that this module uses.

### Depth

Each concept has a depth from 0 to 5. The tutor asks each question at one depth.

| depth | name | a question at this depth asks you to |
|---|---|---|
| 0 | `unseen` | (no question) |
| 1 | `recognise` | pick the correct statement from options |
| 2 | `recall` | state the idea or the rule |
| 3 | `explain` | say why it is true, in your own words |
| 4 | `apply` | use it to solve a problem |
| 5 | `transfer` | use it in a new case or link it to another subject |

The floor is the highest depth that you pass reliably. The ceiling is the lowest depth that you failed (6 until you fail).
The next question goes between them. Thus each answer moves one edge of this range.

These constants name the depths, give the question kind at each depth, and set the goal and the high-confidence line.

In [ ]:
#| export
DEPTHS = ('unseen', 'recognise', 'recall', 'explain', 'apply', 'transfer')
KIND_AT = {1: 'choice', 2: 'recall', 3: 'explain', 4: 'apply', 5: 'transfer'}
DAY = 86400.

### Outcome

Correctness and confidence together tell what an answer means.

| | high confidence | low confidence |
|---|---|---|
| correct | `solid` | `fragile` |
| wrong | `misconception` | `gap` |

"I don't know" is a `gap`. A correct answer that used two or more hints is `fragile`.

In [ ]:
#| export
def outcome(correct:bool, confidence:int=None, hints:int=0, dont_know:bool=False) -> str:
    "One of `solid`, `fragile`, `misconception` or `gap`. With no `confidence`, only the answer and the hints count."
    if dont_know: return 'gap'
    sure = confidence is not None and confidence >= val('checks.high')
    if correct: return 'solid' if (sure or confidence is None) and hints < val('checks.hints') else 'fragile'
    return 'misconception' if sure else 'gap'

A correct answer is `solid` with high confidence. Two hints or low confidence make it `fragile`.

In [ ]:
test_eq(outcome(True, 80), 'solid')
test_eq(outcome(True, 80, hints=2), 'fragile')
test_eq(outcome(True, 30), 'fragile')

With no confidence given, a clean pass is `solid` and a wrong answer is a `gap`, never a `misconception`.

In [ ]:
test_eq([outcome(True), outcome(True, hints=2), outcome(False)], ['solid', 'fragile', 'gap'])

A wrong answer is a `misconception` with high confidence, else a `gap`. "I don't know" is always a `gap`.

In [ ]:
test_eq(outcome(False, 90), 'misconception')
test_eq(outcome(False, 20), 'gap')
test_eq(outcome(True, 90, dont_know=True), 'gap')

### Memory

Recall falls with time as in FSRS: $R(t) = (1 + \frac{19}{81}\frac{t}{S})^{-1/2}$. The time $t$ and the stability $S$ are in days.
At $t = S$, recall is 90%. Thus the due date is the last review plus $S$ days.

`recall` gives the recall now. `grow` gives the new stability after an outcome.

In [ ]:
#| export
def recall(c, now:float=None) -> float:
    "The chance, from 0 to 1, that you still recall concept `c` now."
    if not c.get('stability') or c.get('last') is None: return 0.
    t = (ifnone(now, time.time()) - c['last']) / DAY
    return (1 + 19/81 * max(t, 0) / c['stability']) ** -.5

def grow(s:float, r:float, kind:str) -> float:
    "The new stability after an outcome, from the old stability `s` and the recall `r` at the time of the answer."
    if kind == 'solid': return val('review.solid') if not s else s * (1 + val('review.growth') * (1 - r) + .2)
    if kind == 'fragile': return val('review.shaky') if not s else s * (1 + (1 - r) + .05)
    if kind == 'misconception': return .2
    return max(.2, s * .3)

Recall is 90% after $S$ days, and 0 for a concept with no review.

In [ ]:
test_close(recall(dict(stability=2., last=0.), now=2*DAY), .9, eps=1e-6)
test_eq(recall(dict(stability=0, last=None)), 0.)

A pass grows stability. A harder recall grows it more. A gap shrinks it.

In [ ]:
test_eq(grow(0, 0, 'solid'), 1.)
assert grow(4, .7, 'solid') > grow(4, .99, 'solid') > 4   # a review that was harder to recall teaches more
assert grow(4, .9, 'gap') < 4

### Update

`update` returns the fields to change after one answer at depth `d`.

| outcome | change |
|---|---|
| `solid` | The floor rises to `d`. The streak grows. |
| `fragile` | The floor stays. The concept is `shaky` at `d`, so the next question is at `d` again. |
| `misconception` | The ceiling falls to `d`. The floor falls below `d`. The misconception is kept. |
| `gap` | The ceiling falls to `d`. The floor falls below `d`. |

In [ ]:
#| export
def update(c, depth:int, correct:bool, confidence:int=None, hints:int=0, dont_know:bool=False,
           misconception:str='', now:float=None) -> AttrDict:
    "The state changes for concept `c` after one answer at `depth`, with the outcome in `kind`."
    now, k = ifnone(now, time.time()), outcome(correct, confidence, hints, dont_know)
    fl, ce, st, mi = c.get('floor', 0), c.get('ceiling', 6), c.get('streak', 0), c.get('misses', 0)
    shaky, mis = c.get('shaky'), list(c.get('misconceptions') or [])
    if k == 'solid':
        fl, st, mi, shaky = max(fl, depth), st + 1, 0, None
        if ce <= fl: ce = fl + 1
    elif k == 'fragile': st, mi, shaky = 0, 0, depth if depth > fl else None
    else:
        ce, fl, st, mi, shaky = min(ce, depth), min(fl, depth - 1), 0, mi + 1, None
        if k == 'misconception' and misconception and misconception not in mis: mis = [*mis, misconception][-5:]
    s = grow(c.get('stability', 0), recall(c, now), k)
    return AttrDict(kind=k, floor=max(fl, 0), ceiling=min(ce, 6), streak=st, misses=mi, shaky=shaky, misconceptions=mis,
                    stability=s, last=now, due=now + s * DAY, seen=c.get('seen', 0) + 1)

A `solid` pass at depth 3 on a new concept raises the floor to 3. The review is due in one day (the setting `review.solid`).

In [ ]:
c = AttrDict(floor=0, ceiling=6, stability=0., last=None, streak=0, misses=0, seen=0)
u = update(c, 3, True, 80, now=0.)
test_eq((u.kind, u.floor, u.ceiling, u.streak, u.due), ('solid', 3, 6, 1, DAY))
c.update(u)

A confident wrong answer at depth 4 lowers the ceiling to 4 and keeps the misconception.

In [ ]:
u = update(c, 4, False, 85, misconception='thinks RRF needs normalised scores', now=DAY)
test_eq((u.kind, u.floor, u.ceiling, u.misses), ('misconception', 3, 4, 1))
test_eq(u.misconceptions, ['thinks RRF needs normalised scores'])
c.update(u)

A miss below the floor lowers the floor. A low-confidence pass makes the concept `shaky` at that depth.

In [ ]:
u = update(c, 2, False, 10, now=DAY)    # a miss below the floor: the floor falls
test_eq((u.kind, u.floor, u.ceiling), ('gap', 1, 2))
test_eq(update(c, 4, True, 30).shaky, 4)

### The next depth

Ask in the middle of the range between floor and ceiling, so one answer cuts the range in half.
Ask a `shaky` concept at the same depth again.
When the ceiling is the floor plus 1, ask at the ceiling, because the work is at that edge.

In [ ]:
#| export
def next_depth(c) -> int:
    "The depth for the next question on concept `c`."
    if c.get('shaky'): return c['shaky']
    fl, ce = c.get('floor', 0), min(c.get('ceiling', 6), 6)
    if fl >= 5: return 5
    return min(5, fl + max(1, (ce - fl) // 2))

A new concept starts at depth 3 (explain). An open range gives its middle.

In [ ]:
test_eq(next_depth(dict(floor=0, ceiling=6)), 3)     # a new concept: start at explain, not at the bottom
test_eq(next_depth(dict(floor=3, ceiling=6)), 4)
test_eq(next_depth(dict(floor=0, ceiling=3)), 1)
test_eq(next_depth(dict(floor=1, ceiling=3)), 2)

A closed range gives the ceiling. Depth 5 is the top. A `shaky` depth comes first.

In [ ]:
test_eq(next_depth(dict(floor=2, ceiling=3)), 3)
test_eq(next_depth(dict(floor=5, ceiling=6)), 5)
test_eq(next_depth(dict(floor=1, ceiling=6, shaky=4)), 4)

### What to ask next

`pick` chooses the concept. Code makes the choice, not the model, so the app can show the reason.

1. A `focus` concept, when you chose one and did not learn it yet.
2. A review that is due, with the lowest recall first.
3. The frontier: concepts below the goal with all prereqs at depth 3 or more. Concepts that you started come first.

After two misses in a row on a concept, `pick` goes back to its weakest prereq that is below the goal.

`learned` and `ready` test one concept. `_back` finds the prereq to go back to. After a demo, it waits for the demo ladder (see below).

In [ ]:
#| export
def learned(c, goal:int=None) -> bool:
    "Whether concept `c` is at depth `goal` or more; the default is the setting `checks.goal`."
    return c.get('floor', 0) >= (goal or val('checks.goal'))

def ready(c, by:dict) -> bool:
    "Whether each prereq of `c` that is a known concept is at depth 3 or more."
    return all(by[p].get('floor', 0) >= 3 for p in c.get('prereqs', ()) if p in by)

def _back(c, by, goal):
    "The weakest prereq of `c` that is not learned, after two misses; else `c`. After a demo of `c`, it waits for the third miss since the demo."
    if c.get('misses', 0) < 2: return c, ''
    ladder = val('checks.demo_ladder') and c.get('after_demo') is not None   # the demo module keeps the misses since the demo
    if ladder and c['after_demo'] < 3: return c, ''                          # the demo ladder runs first
    ps = [by[p] for p in c.get('prereqs', ()) if p in by and not learned(by[p], goal)]
    if not ps: return c, ''
    p = min(ps, key=lambda x: x.get('floor', 0))
    return p, f"{'three misses on ' + c['name'] + ' after its demo' if ladder else 'two misses on ' + c['name']}; step back to {p['name']}"

def _gap(c, by):
    "The first gap you named on `c` that is not passed yet, for at most two questions; else None. A gap never blocks `c`."
    return first(by[g] for g in c.get('gaps') or () if g in by and not by[g].get('floor') and by[g].get('seen', 0) < 2)

`pick` returns the concept, the depth and the reason.

In [ ]:
#| export
def pick(concepts, focus:str=None, subject:str=None, goal:int=None, now:float=None, skip=(), near=(), review:bool=True):
    "The next `(concept, depth, why)`, or None when no review is due and the frontier is empty. Concepts in `near` come first on the frontier. `review=False` skips the review of due concepts."
    now, goal = ifnone(now, time.time()), goal or val('checks.goal')
    by = {c['slug']: c for c in concepts}
    pool = [c for c in concepts if c['slug'] not in skip and (not subject or c.get('subject') == subject)]
    def out(c, why):
        rv = 'due for review: ' if why.startswith('due for review') else ''   # a review sent to a prereq or a gap is still a review
        c, back = _back(c, by, goal)
        if (g := _gap(c, by)): return g, min(next_depth(g), g.get('floor', 0) + 1), f"{rv}you said you did not know {g['name']}; it comes before {c['name']}"
        return c, next_depth(c) if not learned(c, goal) else c.get('floor', 0), rv + back if back else why
    if focus and (f := by.get(focus)) and (not learned(f, goal) or (f.get('due') or 0) <= now):
        return out(f, f"you chose {f['name']}")
    due = [c for c in pool if review and c.get('seen') and c.get('due') and c['due'] <= now]
    if due:
        c = min(due, key=lambda c: recall(c, now))
        return out(c, f"due for review: recall about {recall(c, now):.0%}")
    front = [c for c in pool if not learned(c, goal) and ready(c, by)]
    if not front: return None
    c = min(front, key=lambda c: (c['slug'] not in near, -bool(c.get('seen')), c.get('order', 1e9), -c.get('floor', 0)))
    why = 'it is next to the concept you chose' if c['slug'] in near else 'you started it' if c.get('seen') else 'its prereqs are in place'
    return out(c, f'next on the frontier: {why}')

### Strengths and weak spots

`understanding` reads one concept and tells how well you know it: the floor, the ceiling, the recall now, and the reasons that it is weak.
It uses only the state of the concept. Thus a page of all concepts does not need to read the attempts. `weak` is true when a reason says that you
need help now. `strong` needs depth 3 or more, recall of 80% or more, and no reasons.

In [ ]:
#| export
def _is_gap(c): return any(p.get('src') == 'gap' for p in c.get('provenance') or ())

def understanding(c, now:float=None) -> AttrDict:
    "How well you know concept `c`: floor, ceiling, recall, the reasons it is weak, `weak`, `strong`, one `state`, and a score to sort by."
    r, fl, mi, mis = recall(c, now), c.get('floor', 0), c.get('misses', 0), c.get('misconceptions') or []
    why = [*(['you said you did not know it'] if _is_gap(c) and fl < 3 else []),
           *([f'a misconception: {mis[-1]}'] if mis else []),
           *([f'missed the last {mi} times'] if mi >= 2 else []),
           *([f'right, but shaky at the {DEPTHS[c["shaky"]]} level'] if c.get('shaky') else []),
           *([f'recall is down to {r:.0%}'] if c.get('seen') and r < val('review.fading') else [])]   # the state already says fading
    weak = bool(why) or mi > 0
    if c.get('seen', 0) >= 3 and min(c.get('ceiling', 6), 6) - fl >= 3: why.append('its level is not clear yet')
    strong = fl >= 3 and r >= .8 and not mi and not why
    state = ('gap' if _is_gap(c) and fl < 3 else 'strong' if strong else 'weak' if mis or mi or c.get('shaky') else
             'fading' if weak else 'learning' if c.get('seen') else 'unseen')
    return AttrDict(floor=fl, ceiling=c.get('ceiling', 6), recall=r, reasons=why, weak=weak, strong=strong, state=state,
                    score=fl / 5 * r - .2 * len(mis) - .1 * mi)

def weak(c, now:float=None) -> bool: return understanding(c, now).weak

A concept with a misconception and two misses is weak, with both reasons. A deep, fresh concept with no misses is strong.

In [ ]:
now = 1e6
u = understanding(dict(slug='a', floor=1, ceiling=3, misses=2, misconceptions=['scores add'], seen=3, stability=5., last=now), now)
test_eq((u.weak, u.strong, u.reasons[:2]), (True, False, ['a misconception: scores add', 'missed the last 2 times']))
s = understanding(dict(slug='b', floor=4, ceiling=6, seen=5, stability=30., last=now - DAY), now)
test_eq((s.weak, s.strong, s.reasons), (False, True, []))
test_eq(understanding(dict(slug='g', floor=0, provenance=[dict(src='gap')]), now).reasons, ['you said you did not know it'])
assert understanding(dict(slug='f', floor=3, seen=4, stability=1., last=now - 30 * DAY), now).reasons[0].startswith('recall is down to')
test_eq([u.state, s.state, understanding(dict(slug='n')).state], ['weak', 'strong', 'unseen'])

Three concepts: `b` needs `a`. The first pick is `a`, because `b` waits for its prereq.

In [ ]:
def C(slug, **kw): return AttrDict(slug=slug, name=slug, **{**dict(floor=0, ceiling=6, seen=0, misses=0, prereqs=[]), **kw})
cs = [C('a', order=0), C('b', prereqs=['a'], order=1), C('c', order=2)]
test_eq(pick(cs)[0].slug, 'a')                          # b waits for a

A started concept below the goal comes first. When `a` is learned, `b` is ready.

In [ ]:
cs[0].update(floor=3, seen=1, due=10*DAY, stability=10., last=0.)
test_eq(pick(cs, now=DAY)[0].slug, 'a')                 # a was started and is below the goal
cs[0].update(floor=4)
test_eq(pick(cs, now=DAY)[0].slug, 'b')                 # a is learned; b is ready

A due review comes before the frontier, at the floor of the concept.

In [ ]:
r = pick(cs, now=11*DAY)
test_eq((r[0].slug, r[1]), ('a', 4))                    # a is due: a review at its floor
assert 'due for review' in r[2]
# a due concept with two misses goes back to its prereq; the why still says that it is a review, so the daily review limit counts it
r = pick([dict(slug='p', name='P', prereqs=[], floor=0), dict(slug='q', name='Q', prereqs=['p'], floor=2, seen=3, due=1., misses=2)], now=5*DAY)
test_eq(r[0]['slug'], 'p')
assert r[2].startswith('due for review: two misses on Q')

Two misses go back to the weakest prereq. A learned concept that is not due gives no question.

In [ ]:
cs[1].update(misses=2, seen=2, floor=1, ceiling=2); cs[0].update(floor=3, due=None)
test_eq(pick(cs, focus='b', now=DAY)[0].slug, 'a')      # two misses on b: step back to a
test_eq(pick([C('x', floor=4, seen=1, due=1e12)]), None)

After you saw a demo of a concept, `after_demo` on the concept counts your misses since then (the demo module keeps it). While the demo ladder runs, `_back` waits: it steps back only after the third miss since the demo. Here, `b` has 4 misses in all.

The tests below need `set_val`.

In [ ]:
by = {c['slug']: c for c in cs}
test_eq([_back({**cs[1], 'misses': 4, 'after_demo': n}, by, 4)[0]['slug'] for n in (0, 1, 2, 3)], ['b', 'b', 'b', 'a'])
assert 'after its demo' in _back({**cs[1], 'misses': 4, 'after_demo': 3}, by, 4)[1]

With no demo in play, or with the setting `checks.demo_ladder` off, two misses step back, as before.

In [ ]:
test_eq(_back({**cs[1], 'misses': 2, 'after_demo': None}, by, 4)[0]['slug'], 'a')
set_val('checks.demo_ladder', False)
try: test_eq(_back({**cs[1], 'misses': 2, 'after_demo': 1}, by, 4)[0]['slug'], 'a')
finally: set_val('checks.demo_ladder', True)

A concept in `near` comes first on the frontier, before a concept you started elsewhere.

In [ ]:
cs = [C('far', order=0, seen=1), C('next', order=5)]
test_eq(pick(cs)[0].slug, 'far')
r = pick(cs, near={'next'})
test_eq(r[0].slug, 'next')
assert 'next to the concept you chose' in r[2]

A gap that you named on a concept comes first, at the easiest depth. After you pass it, or after two questions on it, the concept comes back.

In [ ]:
nn = dict(slug='neuron', name='Neuron', floor=0, ceiling=6, gaps=['wsum'])
ws = dict(slug='wsum', name='Weighted sum', floor=0, ceiling=6, seen=0)
c, d, why = pick([nn, ws], focus='neuron')
test_eq((c['slug'], d), ('wsum', 1)); assert 'you said you did not know Weighted sum' in why
test_eq(pick([nn, dict(ws, floor=1, seen=1)], focus='neuron')[0]['slug'], 'neuron')
test_eq(pick([nn, dict(ws, seen=2)], focus='neuron')[0]['slug'], 'neuron')

## Ways to learn

The ways you learn: six text ways (code, maths, analogies, worked examples, diagrams and plain words), and demos, ordering, playgrounds, boards and timelines. This module finds which ways work for you, and when to try a new one.

Each question, hint and lesson uses one way to teach: its `method`. The tutor picks the method before it writes the prompt. So the tag is
known, not guessed. When a question closes, its outcome counts for its method: solid 1, shaky 0.5, a miss 0.

The counts give each method a Beta posterior, for each subject and over all subjects. `choose` uses them in two ways:

- Drill: a weak concept gets your best method.
- Explore: a concept that is not weak sometimes gets another method. You set the chance (`explore` in the profile). First `choose` tries
  a method that got better recently. Else it uses a random draw (a Thompson sample).

The counts are in one `stats` document on the shelf. This document is only a cache: `method_stats` builds it again from the attempts.

### The methods

`METHODS` maps each method to the line that the prompt gets. `EXAMPLES_TO` maps the profile's "examples that help" to methods, to set the prior.

In [ ]:
#| export
METHODS = dict(code='Use a short Python code example that shows the idea; keep it under 12 lines.',
               maths='Use equations and formal notation; derive the step, one line at a time.',
               analogy="Use an analogy from the learner's own world (see 'About the learner'), and say where it stops working.",
               example='Use one concrete worked example, step by step, with real numbers or real words.',
               diagram='Describe a picture in words: what to draw, where each part goes, and what moves.',
               words='Use plain words only: no code and no equations.',
               demo='Show a small figure that the learner can change. Ask for a prediction first, then show the result.',
               order='Give the steps of the idea in a mixed order. Ask the learner to put them in the right order.',
               playground='Give a small space where the learner changes one value and sees what happens.',
               board='Give a board with the parts of the idea. Ask the learner to put each part where it belongs.',
               timeline='Give a line of events for the idea. Ask the learner to put the events in time order.')
METHOD_LABELS = dict(code='Code', maths='Maths', analogy='Analogies', example='Worked examples', diagram='Pictures', words='Plain words',
          demo='Demos', order='Put in order', playground='Playgrounds', board='Boards', timeline='Timelines')
EXAMPLES_TO = {'code': 'code', 'maths': 'maths', 'analogies': 'analogy', 'stories': 'analogy', 'diagrams': 'diagram', 'real cases': 'example'}
TEXT_WAYS = ('code', 'maths', 'analogy', 'example', 'diagram', 'words')   # the ways that every session can show
WALK_WAYS = TEXT_WAYS + ('demo', 'order')                                    # the ways that a walk can show now
CREDIT = dict(solid=1., fragile=.5, gap=0., misconception=0.)
FEEDBACK = dict(unclear=('gap', .25), right=('solid', .1))   # a verdict on a demo or a lesson -> the outcome and the weight that it counts for its way
EXPLORE = dict(never=0., sometimes=.15, often=.3)
_METHOD_SRC, _LOCK = _src('stats', 'me'), threading.Lock()

The text of each method is a prompt that you can change in Settings (see `prompts`). `way` gives the text in use: your saved text, else the line in `METHODS`.

In [ ]:
#| export
for _m, _t in METHODS.items(): register(f'way.{_m}', f'Way: {METHOD_LABELS[_m]}', 'How the tutor uses this way in a question, a nudge, a hint and a lesson.', _t)

def way(m:str) -> str:
    "The text of method `m` now: your saved text, else the line in `METHODS`; '' for an unknown method."
    return prompt(f'way.{m}') if m in METHODS else ''

With no saved text, `way` gives the line in `METHODS`.

In [ ]:
test_eq([way(m) for m in METHODS], list(METHODS.values()))
test_eq(way('nope'), '')

### The stats document

`stats` reads the stats document. It gives empty counts when there is no document. `set_stats` writes the document. A lock makes sure that two closes at the same time do not lose a count.

In [ ]:
#| export
EMPTY = dict(methods={}, recent={}, calib={})

@patch
def stats(self:Learning) -> AttrDict:
    "The cached stats: method counts for each subject (`*` is all), recent credits, and calibration."
    r = self.v.by_source(_METHOD_SRC)
    return AttrDict({**EMPTY, **(meta_of(r) if r else {})})

@patch
def set_stats(self:Learning, **kv):
    "Change fields of the stats document."
    m = {**{k: v for k, v in self.stats().items() if k in EMPTY}, **kv}
    r = self.v.put('How you learn: method counts.', title='How you learn', source=_METHOD_SRC, kind='stats', meta=m)

### Credit

`credit` adds one outcome to the counts and returns the new stats. It keeps the last 8 credits of each method, to find a method that gets better.

In [ ]:
#| export
def credit(st, method:str, subject:str, kind:str, w:float=1.) -> dict:
    "The stats `st` with one outcome `kind` added for `method` with the weight `w`, in `subject` and in `*`. Only a full outcome (`w=1`) goes in the recent credits."
    s, ms = CREDIT.get(kind), {k: {m: list(ab) for m, ab in v.items()} for k, v in (st.get('methods') or {}).items()}
    if s is None or method not in METHODS: return dict(st)
    for k in {subject or '*', '*'}:
        a, b = ms.setdefault(k, {}).get(method, [0., 0.])
        ms[k][method] = [a + w * s, b + w * (1 - s)]
    rc = {**(st.get('recent') or {})}
    if w == 1: rc[method] = [*rc.get(method, []), s][-8:]
    return {**st, 'methods': ms, 'recent': rc}

@patch
def credit_method(self:Learning, method:str, subject:str, kind:str, w:float=1.):
    "Count outcome `kind` for `method` with the weight `w` in the stats document."
    with _LOCK: self.set_stats(**{k: v for k, v in credit(self.stats(), method, subject, kind, w).items() if k in EMPTY})

Solid adds 1 to `a`. A miss adds 1 to `b`. Shaky adds half to each. An unknown outcome or method changes nothing.

In [ ]:
st = credit(credit(credit(EMPTY, 'code', 'ml', 'solid'), 'code', 'ml', 'fragile'), 'maths', 'ml', 'gap')
test_eq(st['methods']['ml'], {'code': [1.5, .5], 'maths': [0., 1.]})
test_eq(st['methods']['*'], st['methods']['ml'])
test_eq(st['recent'], {'code': [1., .5], 'maths': [0.]})
test_eq(credit(st, 'code', 'ml', 'closed'), st)

#### Feedback counts a little

The row under a demo or a lesson also counts for the way of that demo or lesson, with a small weight from `FEEDBACK`. "Not clear" adds 0.25 of a miss, so four of them count as one miss, and one click does not decide the way. "Just right" adds 0.1 of a pass. "Too simple" adds nothing: the way worked, but the step was too small. A part of an outcome does not go in the recent credits, so it does not make a way look better lately.

In [ ]:
st = credit(EMPTY, 'analogy', 'ml', *FEEDBACK['unclear'])
test_eq(st['methods']['ml']['analogy'], [0., .25])
test_eq(st['recent'], {})
test_eq(credit(st, 'analogy', 'ml', *FEEDBACK['right'])['methods']['ml']['analogy'], [.1, .25])

### The posterior

`posterior` gives `(a, b)` for a method: Beta(1, 1), plus 2 on `a` for a method in the profile's examples. For a subject, it adds the subject counts and the `*` counts. It scales the `*` counts down to at most 4 pseudo-counts. So a new subject starts from what works for you in other subjects.

In [ ]:
#| export
def posterior(st, m:str, subject:str='', prior=()) -> tuple:
    "The Beta `(a, b)` of method `m` in `subject`."
    ms = st.get('methods') or {}
    a, b = 1. + 2 * (m in prior), 1.
    g = (ms.get('*') or {}).get(m, [0., 0.])
    if subject and (s := (ms.get(subject) or {}).get(m)):
        k = min(1., 4 / max(sum(g), 1e-9))
        return a + s[0] + k * g[0], b + s[1] + k * g[1]
    return a + g[0], b + g[1]

def mean(ab) -> float: return ab[0] / sum(ab)

def improving(st, m:str) -> tuple|None:
    "`(right, n)` over the last 5 credits of `m`, when they are at least 0.2 above its mean over all; else None."
    r, ab = (st.get('recent') or {}).get(m, []), (st.get('methods') or {}).get('*', {}).get(m)
    if len(r) < 4 or not ab or sum(ab) < 4: return None
    last = r[-5:]
    return (round(sum(last)), len(last)) if sum(last) / len(last) >= ab[0] / sum(ab) + .2 else None

With no counts, the prior decides. `improving` finds a method that got better recently.

In [ ]:
test_eq(posterior(EMPTY, 'code'), (1., 1.))
test_eq(posterior(EMPTY, 'code', prior=['code']), (3., 1.))
gm = dict(methods={'*': {'maths': [2., 4.]}}, recent={'maths': [0., 0., 1., 1., 1., 1.]})
test_eq(improving(gm, 'maths'), (4, 5))
test_is(improving(dict(gm, recent={'maths': [1., 0., 0., 0.]}), 'maths'), None)

### Choose a method

`choose` returns `(method, note)`. The note goes in the line that says why you get this question.

In [ ]:
#| export
def _count(st, m, subject=''):
    "The raw `(right, n)` of method `m`, in `subject` when it has counts there, else over all."
    ms = st.get('methods') or {}
    a, b = (ms.get(subject) or {}).get(m) or (ms.get('*') or {}).get(m) or [0., 0.]
    return round(a), round(a + b)

def choose(st, subject:str='', weak:bool=False, rng=None, explore:str='sometimes', prior=(), allowed=None) -> tuple:
    "The method for the next question, and a note on why, from the `allowed` ways (default `TEXT_WAYS`). A weak concept gets the best method; else, sometimes, another one."
    rng = rng or random
    cands = [m for m in allowed or TEXT_WAYS if m in METHODS] or list(TEXT_WAYS)
    ps = {m: posterior(st, m, subject, prior) for m in cands}
    best = max(ps, key=lambda m: (mean(ps[m]), m in prior))
    if weak:
        k, n = _count(st, best, subject)
        return best, f"drill with {METHOD_LABELS[best].lower()}: {k} of {n}, your best way" if n else (f"drill with {METHOD_LABELS[best].lower()}: a way that you said helps" if best in prior else '')
    if (up := first((m, improving(st, m)) for m in cands if m != best and improving(st, m))):
        m, (k, n) = up
        if rng.random() < max(EXPLORE.get(explore, .15) * 2, .3) * bool(EXPLORE.get(explore, .15)):
            return m, f"trying {METHOD_LABELS[m].lower()}: you got better with it ({k} of {n} lately)"
    if (others := [m for m in cands if m != best]) and rng.random() < EXPLORE.get(explore, .15):
        m = max(others, key=lambda m: rng.betavariate(*ps[m]))
        return m, f"trying {METHOD_LABELS[m].lower()}: a new way to learn this"
    return best, ''

A weak concept gets the best method. With `explore='never'`, `choose` always gives the best method. With `sometimes`, it tries a method that got better at times. It still gives the best method most of the time.

In [ ]:
cs = dict(methods={'*': {'code': [7., 2.], 'maths': [2., 4.]}}, recent={'maths': [0., 0., 1., 1., 1., 1.]})
m, note = choose(cs, 'ml', weak=True)
test_eq(m, 'code'); assert note.startswith('drill with code: 7 of 9')
test_eq({choose(cs, 'ml', rng=random.Random(i), explore='never')[0] for i in range(30)}, {'code'})
picks = [choose(cs, 'ml', rng=random.Random(i)) for i in range(60)]
assert any(n.startswith('trying maths: you got better') for _, n in picks)
test_eq(sum(m == 'code' for m, _ in picks) > 30, True)   # with `sometimes`, the best method still comes most of the time

### The ways that fit

A session cannot show every way. `choose` takes `allowed`: the ways it may pick from. The default is `TEXT_WAYS`, the ways that every session can show. A list with no known way gives `TEXT_WAYS` too.

In [ ]:
rng = random.Random(0)
assert {'demo', 'order', 'playground', 'board', 'timeline'} <= set(METHODS) and 'way.timeline' in PROMPTS
assert all(choose(EMPTY, rng=rng)[0] in TEXT_WAYS for _ in range(200))          # a session never gets a way it cannot show
assert all(choose(EMPTY, rng=rng, allowed=['demo', 'maths'])[0] in ('demo', 'maths') for _ in range(200))
assert choose(EMPTY, rng=rng, allowed=['board'])[0] in METHODS                   # an allowed list keeps its own ways
assert choose(EMPTY, rng=rng, allowed=['nope'])[0] in TEXT_WAYS                  # no known way: back to the text ways

### The way for a check

`check_for` picks the way for one check of one idea. It looks only at the ways of the idea that a walk can show. It does not repeat the way of the last check when it has another. The kind of the check follows the way, and the depth follows the kind. A short check is depth 3 or more: it asks for the idea and why it holds, never for a fact to recall. It is depth 4 when the floor is 3 or more.

A hands-on way of the idea (a key of `CHECK_KIND`: demo, order or code) comes before its text ways. The model lists a hands-on way only for an idea that fits it: a process, a sequence, or code. A code check is a notebook that runs in Leela. Without this rule, the learner stats and the profile pick a text way almost every time, because the profile names only text ways. Then nearly every check is a short answer.

Only the idea's own ways can give a hands-on check. An idea with no way that a walk can show gets a short answer in a text way. A walk passes `hands=False` when it has its share of hands-on checks (see `hands_most` in `journey`).

In [ ]:
#| export
CHECK_KIND = dict(demo='demo', order='order', code='notebook')   # the code way is a notebook check: it runs in Leela
DEPTH = dict(short=3, demo=2, order=2, notebook=4)

def check_for(c, st, rng=None, explore:str='sometimes', prior=(), last:str='', hands:bool=True) -> AttrDict:
    "The way, kind, depth and reason for one check of concept `c`. A hands-on way of the idea comes before its text ways; `hands=False` leaves the hands-on ways out."
    own = [w for w in c.get('ways') or () if w in WALK_WAYS]
    ways = [w for w in own if w != last] or own
    hand = [w for w in ways if w in CHECK_KIND] if hands else []
    text = [w for w in ways if w not in CHECK_KIND] or [w for w in own if w not in CHECK_KIND] or [w for w in TEXT_WAYS if w not in CHECK_KIND]
    way, note = choose(st, c.get('subject', ''), False, rng, explore, prior, allowed=hand or text)
    kind = CHECK_KIND.get(way, 'short')
    depth = (DEPTH['short'] if (c.get('floor') or 0) < 3 else 4) if kind == 'short' else DEPTH[kind]
    return AttrDict(way=way, kind=kind, depth=depth, why=note or f"{METHOD_LABELS[way].lower()} {'fit' if way == 'words' else 'fits'} this idea")

A check takes a way of its idea. An idea with no way that a walk can show gets a text way.

In [ ]:
c = AttrDict(ways=['code', 'maths'], floor=0)
k = check_for(c, EMPTY, rng=random.Random(1))
assert (k.way, k.kind, k.depth) in {('code', 'notebook', 4), ('maths', 'short', 3)} and k.why
assert check_for(AttrDict(ways=['demo'], floor=0), EMPTY).kind == 'demo'
assert check_for(AttrDict(ways=['timeline'], floor=3), EMPTY).way in TEXT_WAYS   # no timeline view yet
assert check_for(AttrDict(ways=['words'], floor=0), EMPTY).depth == 3   # a new idea: say the idea and why, not a fact
assert check_for(AttrDict(ways=['words'], floor=3), EMPTY).depth == 4
assert all(check_for(AttrDict(ways=['demo', 'code'], floor=0), EMPTY, rng=rng, last='demo').way == 'code' for _ in range(20))

An idea that lists a hands-on way (demo, order or code) gets a check of that kind. This is true when a text way comes first in the list,
and when the profile likes a text way. An idea with only text ways still gets a short answer.

In [ ]:
assert all(check_for(AttrDict(ways=['words', 'example', 'order'], floor=0), EMPTY, random.Random(k), prior=['example']).kind == 'order' for k in range(50))
assert all(check_for(AttrDict(ways=['maths', 'demo'], floor=0), EMPTY, random.Random(k)).kind == 'demo' for k in range(50))
assert all(check_for(AttrDict(ways=['words', 'analogy'], floor=0), EMPTY, random.Random(k)).kind == 'short' for k in range(50))

An idea with no ways, or with no way that a walk can show, gets a short answer, never a code check. With `hands=False` (the walk has its share of hands-on checks), an idea gets a short answer in one of its text ways, else in any text way.

In [ ]:
assert all(check_for(AttrDict(ways=w, floor=0), EMPTY, random.Random(k)).kind == 'short' for w in ([], None, ['timeline']) for k in range(50))
assert all(check_for(AttrDict(ways=['code', 'maths'], floor=0), EMPTY, random.Random(k), hands=False).way == 'maths' for k in range(20))
assert all(check_for(AttrDict(ways=['demo'], floor=0), EMPTY, random.Random(k), hands=False).kind == 'short' for k in range(20))

### In plain words, and from the attempts

`methods_text` says what works, for a prompt and for the page. `method_stats` builds the counts again from the attempts, in order.

In [ ]:
#| export
def methods_text(st, subject:str='', n:int=3) -> str:
    "The methods with counts, best first, as one line; empty when nothing is counted."
    ms = {m: ab for m, ab in ((st.get('methods') or {}).get(subject or '*') or {}).items() if sum(ab) >= 1}
    if not ms: return ''
    top = sorted(ms, key=lambda m: -mean(posterior(st, m, subject)))[:n]
    bits = [f"{METHOD_LABELS[m].lower()} {round(ms[m][0])} of {round(sum(ms[m]))}" + (' (better lately)' if improving(st, m) else '') for m in top]
    return 'Ways that work for you: ' + '; '.join(bits) + '.'

@patch
def method_stats(self:Learning, n:int=2000) -> dict:
    "Count every final attempt with a method again, oldest first, with the feedback on each attempt, and save the counts. Return the stats."
    st = {**EMPTY, 'calib': self.stats().get('calib') or {}}
    for a in reversed(self.attempts(limit=n)):
        if a.get('final') and a.get('method') and not a.get('dont_know'):
            c = self.concept(a.concept)
            st = credit(st, a.method, (c or {}).get('subject', ''), a.get('outcome', ''))
        for f in a.get('fb') or ():   # the verdicts on the demo or the lesson of this attempt
            if f.get('verdict') in FEEDBACK and f.get('way'): st = credit(st, f['way'], (self.concept(a.concept) or {}).get('subject', ''), *FEEDBACK[f['verdict']])
    with _LOCK: self.set_stats(**st)
    return st

The counts that `credit_method` keeps match a rebuild from the attempts.

In [ ]:
ml_ = temp_learning(); ml_.put('Ranking', 'Order items by a score.', 'ml')
for m, k in (('code', 'solid'), ('code', 'fragile'), ('maths', 'gap')):
    ml_.record('ranking', f'{m} {k}?', final=True, method=m, outcome=k); ml_.credit_method(m, 'ml', k); time.sleep(.01)
inc = ml_.stats()
test_eq(ml_.method_stats()['methods'], inc.methods)
assert methods_text(inc, 'ml').startswith('Ways that work for you: code 2 of 2')

## The profile

What abhyasa knows about you: your background, goals, interests, the examples that help you, your pace, and what confuses you.

The tutor asks better questions when it knows the learner. A coder gets code examples. A Sanskrit reader gets a Sanskrit example.
This module keeps one `profile` document on the learning shelf. The profile gets data in three ways: a short first conversation, edits by the
learner, and changes that the tutor makes from evidence. The module keeps each change with who made it and why. So the learner can see
each change and undo it. The profile stays in the vault on this machine.

### The fields

A field is a list of short items, or one value. `PACES` names the three paces. `EXAMPLES` names the kinds of example a learner can like.

In [ ]:
#| export
LEARNING = ('pace', 'session_len', 'explore')   # the three values that the Settings page shows too: their defaults and ranges are in `settings.SETTINGS`
PROFILE = dict(background=[], goals=[], interests=[], examples=[], confuses=[], notes='', **{k: SETTINGS[f'learning.{k}'].dflt for k in LEARNING})
LIST_FIELDS = ('background', 'goals', 'interests', 'examples', 'confuses')
EXAMPLES = ('code', 'maths', 'analogies', 'stories', 'diagrams', 'real cases')
PACES = dict(zip(SETTINGS['learning.pace'].choices, ('small steps: at most one depth above what you know', 'the usual steps', 'bigger steps, further past what you know')))
PROFILE_LABELS = dict(background='Background', goals='Goals', interests='Interests', examples='Examples that help', pace='Pace',
              session_len='Questions in a session', confuses='What confuses you', notes='Notes', explore='How often to try a new way')
_PROFILE_SRC = _src('profile', 'me')

`profile_text` gives the profile as lines for a prompt. It is empty when the profile has no data. So the prompt for a new learner does not change.

In [ ]:
#| export
def profile_text(p) -> str:
    "The profile as prompt lines; empty when nothing is known."
    out = [f"{PROFILE_LABELS[k]}: {'; '.join(p[k])}" for k in LIST_FIELDS if p.get(k)]
    if p.get('pace', 'steady') != 'steady': out.append(f"Pace: {p['pace']} ({PACES[p['pace']]})")
    if p.get('notes'): out.append(f"Notes: {p['notes']}")
    return '\n'.join(out)

`clean` makes a value fit its field. A list field takes a list or lines of text. The pace must be a known pace. The session length stays between 3 and 40.

In [ ]:
#| export
def clean(k, v):
    "The value `v` made to fit field `k`."
    if k in LIST_FIELDS:
        if isinstance(v, str): v = v.splitlines()
        return list(dict.fromkeys(x.strip(' -*\t') for x in v or () if x and x.strip(' -*\t')))
    if k in LEARNING:   # the row of the settings table clamps a number and checks a choice; a form sends a number as text
        if k == 'session_len' and isinstance(v, str) and v.strip().lstrip('-').isdigit(): v = int(v)
        return fit(SETTINGS[f'learning.{k}'], v)
    return str(v or '').strip()

### The document

`profile` reads the document. It uses the defaults for the fields that are not set. `known` is False until the profile has a saved value.

In [ ]:
#| export
@patch
def profile(self:Learning) -> AttrDict:
    "The learner profile, with its list of changes, newest first."
    r = self.v.by_source(_PROFILE_SRC)
    m = meta_of(r) if r else {}
    return AttrDict({**PROFILE, 'changes': [], **{k: v for k, v in m.items() if k in (*PROFILE, 'changes', 'at')}, 'known': bool(r)})

`set_profile` changes fields. It keeps each change with the old value, the new value, who made it (`you`, `onboarding` or `tutor`) and why.

In [ ]:
#| export
@patch
def set_profile(self:Learning, by:str='you', why:str='', **kv) -> list:
    "Change fields of the profile and keep each change with who made it and why. Return the changes."
    p, ch = self.profile(), []
    for k, v in kv.items():
        if k not in PROFILE: continue
        v = clean(k, v)
        if v != p[k]: ch.append(dict(id=uuid.uuid4().hex[:10], at=time.time(), field=k, old=p[k], new=v, by=by, why=why)); p[k] = v
    if not ch: return []
    m = {**{k: p[k] for k in PROFILE}, 'changes': [*ch[::-1], *p.changes][:200], 'at': time.time()}
    r = self.v.put(profile_text(p) or 'An empty profile.', title='About you', source=_PROFILE_SRC, kind='profile', meta=m)
    return ch

`undo_profile` puts back the old value of one change. It finds the change by its id. Two changes in one save can have the same time, so the time can not tell them apart. A change from before the ids has none, so for it the exact time is used. The undo is a change too, so the list shows all the changes.

In [ ]:
#| export
@patch
def undo_profile(self:Learning, ref) -> list:
    "Put back the old value of the change `ref`: its id, or for an older change with no id, its exact time `at`."
    cs = self.profile().changes
    c = first(c for c in cs if c.get('id') == str(ref))
    if c is None:
        try: c = first(c for c in cs if not c.get('id') and c['at'] == float(ref))
        except (TypeError, ValueError): c = None
    return self.set_profile(by='you', why='you undid a change', **{c['field']: c['old']}) if c else []

@patch
def clear_profile(self:Learning):
    "Remove the profile and its changes."
    if (r := self.v.by_source(_PROFILE_SRC)): self.v.forget(r['id'])

### The first conversation

The learner answers four questions, by voice or by text. One model call turns the answers into fields.

In [ ]:
#| export
ONBOARD_QS = (('who', 'What do you do, and what do you know well already?', 'For example: I write Python each day. I know Sanskrit grammar.'),
              ('want', 'What do you want to learn, and why?', 'For example: how LLMs work, so I can read the papers.'),
              ('how', 'What kind of explanation works for you?', 'Code, maths, analogies, stories, pictures, real cases.'),
              ('time', 'How long do you like to study at one time? Small steps or big ones?', 'For example: twenty minutes, small steps.'))

ONBOARD_SP = register('profile.onboard', 'Profile from your first answers', 'How the tutor makes your profile from your answers to the four first questions.', """You make a short learner profile from what a learner says about themselves.
Use only what they say; leave a field empty when they say nothing about it. Keep each item under 12 words, in their words where you can.""")

def onboard_prompt(answers:dict) -> str:
    "The prompt that turns the onboarding answers into profile fields."
    qa = '\n'.join(f"Q: {q}\nA: {answers.get(k) or '(no answer)'}" for k, q, _ in ONBOARD_QS)
    return (f"{qa}\n\nReturn JSON: {{\"background\": [str], \"goals\": [str], \"interests\": [str], "
            f"\"examples\": [str, each one of {list(EXAMPLES)}], \"pace\": \"gentle\" | \"steady\" | \"fast\", "
            '"session_len": int (questions in one sitting; one question takes about two minutes), "notes": str}')

def onboard(lr:Learning, ask, answers:dict) -> list:
    "Fill the profile from the onboarding answers. Return the changes."
    from abhyasa.models import ask_json, for_task   # here: the model layer imports this module
    r = ask_json(for_task(ask, 'profile'), onboard_prompt(answers), prompt('profile.onboard'))
    return lr.set_profile(by='onboarding', why='from what you said when you started', **{k: v for k, v in r.items() if k in PROFILE})

### The tutor learns from evidence

`observe` reads the recent answers: right or not, confidence, hints, misconceptions and think-aloud. A small model call proposes at
most three changes, each with its reason. `observe` applies them immediately and keeps each one with its reason. So the learner sees them and can undo them.

In [ ]:
#| export
OBSERVE_SP = register('profile.observe', 'Profile from your answers', 'How the tutor changes your profile from your recent answers.', """You keep a short profile of one learner. You change it only from evidence in their recent answers.
Propose at most 3 changes, and none when the evidence is weak. Good changes: a topic that confuses them again and again,
a kind of example that works (they explain well with code or analogies), a pace that fits (many hints and misses: gentle;
quick solid passes: fast), background that their answers show. Each `why` names the evidence in one short sentence, to the
learner as "you", in ASD-STE100 Simplified Technical English.""")

def evidence(lr:Learning, n:int=12) -> str:
    "The last `n` final answers as short lines for a prompt: the outcome, the method, the first error, gaps you named, lessons you asked for, and what you said."
    def line(a):
        c = lr.concept(a.concept)
        th = a.get('thread') or []
        first_err = first(t.get('first_error') for t in th if t.get('kind') == 'nudge' and t.get('first_error'))
        said = [t['text'][:160] for t in th if t.get('who') == 'you' and t.get('text')]
        bits = [f"{c.name if c else a.concept} at depth {a.depth}: {a.get('outcome') or ('right' if a.correct else 'wrong')}",
                a.get('method') and f"taught with {a.method}", f"confidence {a.confidence}", f"{a.hints} hints" if a.hints else '',
                first_err and f"first error: {first_err}", a.get('gaps') and f"said they did not know: {', '.join(a.gaps)}",
                a.get('taught') and f"asked for a lesson after it ({a.get('teach_method') or 'same way'})",
                a.misconception and f"misconception: {a.misconception}",
                f"what they said: {' / '.join(said)}" if said else a.answer and f"answer: {a.answer[:160]}", a.thinking and f"think-aloud: {a.thinking[:160]}"]
        return '- ' + '; '.join(b for b in bits if b)
    return '\n'.join([line(a) for a in lr.attempts(limit=n * 3) if a.get('final') and a.get('kind') != 'demo'][:n])

def observe_prompt(p, ev:str, st=None) -> str:
    "The prompt that proposes changes to profile `p` from the evidence `ev` and the method counts in the stats `st`."
    return (f"The profile now:\n{profile_text(p) or '(empty)'}\n\nTheir recent answers, newest first:\n{ev}\n\n"
            + (f"{mt}\n\n" if (mt := methods_text(st or {})) else '')
            + 'Return JSON: {"changes": [{"field": "background" | "interests" | "examples" | "pace" | "confuses", '
            '"add": str or null, "remove": str or null, "value": str or null (for pace), "why": str}]}')

`observe` applies the changes. A list field takes `add` and `remove`. The pace takes `value`. The goals and the notes are for the learner only, so `observe` drops a change to them.

In [ ]:
#| export
def observe(lr:Learning, ask, n:int=12) -> list:
    "Change the profile from the recent answers. Return the changes, each with its reason."
    from abhyasa.models import ask_json, for_task   # here: the model layer imports this module
    if not (ev := evidence(lr, n)): return []
    lr.set_stats(calib=dict(calibration(lr)))   # cached here, every few closes, so a question does not read all attempts
    r = ask_json(for_task(ask, 'profile'), observe_prompt(lr.profile(), ev, lr.stats()), prompt('profile.observe'))
    out = []
    for c in (r.get('changes') or [])[:3] if isinstance(r, dict) else []:
        f, p = c.get('field'), lr.profile()
        if f in ('background', 'interests', 'examples', 'confuses'): new = [x for x in p[f] if x != c.get('remove')] + ([c['add']] if c.get('add') else [])
        elif f == 'pace' and c.get('value'): new = c['value']
        else: continue
        out += lr.set_profile(by='tutor', why=c.get('why', ''), **{f: new})
    return out

### Who the words are for

Each text that explains an idea is for a human, and humans differ. One learner is new to the idea, and another knows it well.
A text that only says the definition again teaches little to both. So each teaching prompt gets two more parts:

- `teach.human`, one shared prompt with the rules on how to teach. It comes from the research on how people learn: adapt to the level
  (the expertise reversal effect), a concrete case before the abstract rule (concreteness fading), worked steps for a new learner,
  an analogy with its limits, the reason why, and the common wrong idea.
- `level_line`, one short line on how well this learner knows the idea now. It comes from the learner model: the state of the idea
  (`understanding`), the reasons it is weak, the needs that are not in place, and the profile.

The rules say what to do for each level. Shaky, a gap or a missing need: keep it simple, one idea at a time, a concrete case first,
step by step. Solid: be short, with the exact terms and fewer worked steps.

In [ ]:
#| export
HUMAN = register('teach.human', 'Who the words are for', 'The rules on how to teach that the tutor adds to each prompt that explains an idea. The line on your level comes after them.', """The reader is a human who wants to understand the idea, not only to read its definition. Learners differ: one is new to the idea, another knows it well. The level line in the prompt says how well this learner knows it.
- Make the idea make sense. Say why it works, or what problem it solves. Never only say the definition again in other words.
- Start from something that the learner knows. Then give one concrete case, with small real numbers or everyday things, before the general rule.
- An analogy can help. When you use one, say in a few words where it stops being true.
- When a wrong idea is common, name it and say why it is wrong.
- Keep it simple when the level line says new, shaky or a gap, or names a need that is not in place. Teach one idea at a time. Give the concrete case first, then take the learner through it step by step. Say each new term in plain words before you use it. Give a missing need in one plain line before you build on it. In a lesson or a demo, check that they follow before you go on.
- Be short when the level line says solid. Use the exact terms. Give the key step and a harder case or an edge case. Do not explain the steps that they know.
- With no level line, write for a reader who is new to the idea, and still give the exact terms.
- Plain words are not literal words. A short sentence can carry an example, a picture in words, or the reason why.
- Name a technical thing by its real name. Say in plain words what it is the first time. Never make up a nickname or a metaphor for it.
- Write in ASD-STE100 Simplified Technical English at a US grade 9 reading level: sentences of at most 20 words, common words, the active voice, no semicolons.
- These rules do not change the rules on sources and facts. A quote stays word for word. Say only what is true.""")

def teach_sp(name:str) -> str:
    "The prompt `name` with the shared rules on how to teach (`teach.human`) after it."
    return f"{prompt(name)}\n\n{prompt('teach.human')}"

`level_line` turns the learner model into one short line. A need is a prereq or a gap that is not at depth 3 yet; a walk can give its own list in `missing`. The profile adds the background, the examples that help, the pace and what confuses them.

In [ ]:
#| export
LEVELS = dict(unseen='new to this idea', gap='a gap: they said that they did not know it', weak='shaky', fading='fading: they learned it, and recall is going down',
              learning='learning it', strong='solid')

def level_line(c, by:dict=None, p=None, missing=None) -> str:
    "One short line on how well the learner knows concept `c` now, for a teaching prompt: the state and why, the needs that are not in place, the profile, and what to do. '' with no concept."
    if not c: return ''
    u = understanding(c)
    out = [f"The learner's level on this idea: {LEVELS[u.state]}" + (f", at the {DEPTHS[u.floor]} level" if u.floor and u.state in ('learning', 'strong') else '')
           + (f" ({'; '.join(u.reasons)})" if u.reasons else '') + '.']
    if missing is None: missing = [by[s]['name'] for s in dict.fromkeys([*(c.get('prereqs') or ()), *(c.get('gaps') or ())]) if by and s in by and by[s].get('floor', 0) < 3]
    if missing: out.append(f"Needs that are not in place yet: {'; '.join(missing[:3])}.")
    p = p or {}
    bits = [p.get('background') and f"Background: {'; '.join(p['background'][:2])}", p.get('examples') and f"Examples that help: {', '.join(p['examples'][:3])}",
            p.get('pace', 'steady') != 'steady' and f"Pace: {p['pace']}", p.get('confuses') and f"What confuses them: {'; '.join(p['confuses'][:2])}"]
    out += [f'{b}.' for b in bits if b]
    simple = u.state in ('unseen', 'gap', 'weak') or bool(missing) or p.get('pace') == 'gentle'
    out.append('So: keep it simple.' if simple else 'So: be short.' if u.state == 'strong' else 'So: the usual steps.')
    return ' '.join(out)

@patch
def level_of(self:Learning, c, missing=None, profile:bool=True) -> str:
    "The level line of concept `c` (a slug or a dict) for this learner, with the needs on the shelf and, unless `profile=False`, the profile. A question takes no profile: it uses the real terms of the subject."
    by = {x.slug: x for x in self.concepts()}
    return level_line(by.get(c) if isinstance(c, str) else c, by, self.profile() if profile else {}, missing)

A new idea with no profile asks for the simple form.

In [ ]:
test_eq(level_line(dict(name='Dot product')), "The learner's level on this idea: new to this idea. So: keep it simple.")
test_eq(level_line(None), '')

A solid idea asks for the short form, with its depth.

In [ ]:
import time as _t
solid = dict(name='Dot product', floor=4, seen=5, stability=30, last=_t.time())
test_eq(level_line(solid), "The learner's level on this idea: solid, at the apply level. So: be short.")

A need that is not in place makes it simple, even for an idea that is not weak. The profile adds its lines.

In [ ]:
by = dict(dot=dict(slug='dot', name='Dot product', floor=1))
l = level_line(dict(name='Attention', prereqs=['dot'], floor=3, seen=2, stability=30, last=_t.time()), by, dict(background=['writes Python'], pace='gentle'))
assert 'Needs that are not in place yet: Dot product.' in l and 'Background: writes Python.' in l and l.endswith('So: keep it simple.')

A gap that the learner named, and an idea with misses, say why they are weak.

In [ ]:
assert 'a gap' in level_line(dict(name='Softmax', provenance=[dict(src='gap')]))
assert 'shaky (missed the last 2 times' in level_line(dict(name='Softmax', floor=1, seen=3, misses=2))

`teach_sp` puts the shared rules after a prompt.

In [ ]:
assert teach_sp('profile.observe').endswith(prompt('teach.human')) and 'Never only say the definition again' in HUMAN

`plainly` makes a text that the tutor wrote plain. It is rupaka's `plain` when the setting `checks.readable` is on: a text that fails rupaka's `plain_why` is written again once, and the new text stays only when it is better. Each place that shows a model text to the learner uses it.

In [ ]:
#| export
from rupaka.agent import plain

def plainly(ask, x, terms=(), keep=()):
    "`x` (a text, or a dict of texts) through rupaka's `plain` with the model `ask`, when `checks.readable` is on; else `x`."
    return plain(ask, x, terms, keep) if val('checks.readable') else x

A plain text makes no model call. A hard text makes one call, and the plainer text stays. With the setting off, nothing changes.

In [ ]:
calls = []
def fake_ask(p, sp=''): calls.append(p); return 'In a DDPM, the amount of noise at each step is set before training. Which DDPM rule breaks when the network learns it?'
hard = 'You ask the network to learn the noise recipe as it trains. Why does this depart from a DDPM simplifying assumption?'
test_eq(plainly(fake_ask, 'The cat sat.'), 'The cat sat.'); test_eq(calls, [])
assert plainly(fake_ask, hard).startswith('In a DDPM') and len(calls) == 1
set_val('checks.readable', False)
try: test_eq(plainly(fake_ask, hard), hard)
finally: set_val('checks.readable', True)

### How well you know what you know

A good learner knows what they know. When you feel sure and you are often wrong, you stop checking your own thinking. When you feel unsure and you are often right, you hold back ideas that work. The "Sure?" choice on each answer gives the data. `calibration` counts the first try of each question. It counts how many times you felt sure (60% or more), and how many of those answers were right. It counts the same for unsure. An answer with no confidence, a "don't know" and a demo do not count.

In [ ]:
#| export
def calibration(lr:Learning, n:int=300) -> AttrDict:
    "How often the first try of a question was right, when you felt sure and when you felt unsure."
    seen, r, high = set(), AttrDict(sure=0, sure_right=0, unsure=0, unsure_right=0), val('checks.high')
    for a in reversed(lr.attempts(limit=n)):   # oldest first, so the first try of each question counts
        k = (a.get('session'), a.question)
        if k in seen: continue
        seen.add(k)
        if a.get('kind') == 'demo' or a.get('dont_know') or a.get('confidence') is None: continue
        b = 'sure' if a.confidence >= high else 'unsure'
        r[b], r[f'{b}_right'] = r[b] + 1, r[f'{b}_right'] + bool(a.correct)
    return r

`calibration_text` gives the calibration in plain words, with one line about what it means. With fewer than five answers, it gives an empty text.

In [ ]:
#| export
def calibration_text(r) -> str:
    "The calibration `r` in plain words; empty with fewer than five answers."
    if r.sure + r.unsure < 5: return ''
    out = [f'When you felt sure, you were right in {r.sure_right} of {r.sure}.' if r.sure else '',
           f'When you felt unsure, you were right in {r.unsure_right} of {r.unsure}.' if r.unsure else '']
    if r.sure >= 3 and r.sure_right / r.sure < .7: out.append('You are often sure and wrong. Before you answer, ask: how could this be wrong?')
    elif r.unsure >= 3 and r.unsure_right / r.unsure > .7: out.append('You know more than you think. Trust your first idea a little more.')
    elif r.sure >= 3: out.append('When you feel sure, you are usually right. Your sense of what you know is good.')
    return ' '.join(o for o in out if o)

### Tests

A new shelf has an empty profile, and its prompt text is empty.

In [ ]:
from abhyasa.models import Fake

In [ ]:
lr = temp_learning()
p = lr.profile()
test_eq((p.known, p.pace, p.background, profile_text(p)), (False, 'steady', [], ''))

`set_profile` keeps each change, with who made it. Lines of text become a list.

In [ ]:
ch = lr.set_profile(background='codes in Python\nknows Sanskrit grammar', pace='gentle')
test_eq([c['field'] for c in ch], ['background', 'pace'])
p = lr.profile()
test_eq((p.known, p.background, p.changes[0]['by']), (True, ['codes in Python', 'knows Sanskrit grammar'], 'you'))
assert 'Background: codes in Python; knows Sanskrit grammar' in profile_text(p) and 'Pace: gentle' in profile_text(p)

When a value stays the same, `set_profile` makes no change. A bad pace becomes `steady`, and the session length stays in its range.

In [ ]:
test_eq(lr.set_profile(pace='gentle'), [])
test_eq((clean('pace', 'warp'), clean('session_len', '200'), clean('session_len', 'x')), ('steady', 40, 12))

`undo_profile` puts back the old value, and keeps the undo as a change.

In [ ]:
cid = [c for c in lr.profile().changes if c['field'] == 'pace'][0]['id']
lr.undo_profile(cid)
test_eq((lr.profile().pace, lr.profile().changes[0]['why']), ('steady', 'you undid a change'))

Two changes in one save can have the same time. Each change has its own id, so the undo changes only the one that you pick.

In [ ]:
import time as _t

In [ ]:
real = _t.time
_t.time = lambda: 1000.0   # every change of this save gets the same time
try: lr.set_profile(pace='fast', session_len=20)
finally: _t.time = real
two = lr.profile().changes[:2]
test_eq(two[0]['at'], two[1]['at'])
lr.undo_profile([c for c in two if c['field'] == 'pace'][0]['id'])
test_eq((lr.profile().pace, lr.profile().session_len), ('steady', 20))

`onboard` fills fields from the answers with one model call.

In [ ]:
fake = Fake({'make a short learner profile': dict(goals=['understand LLM papers'], examples=['code', 'maths'], session_len=8, pace='steady', notes='')})
ch = onboard(lr, fake, dict(want='LLMs, to read papers', how='code please'))
p = lr.profile()
test_eq((p.goals, p.examples, p.session_len), (['understand LLM papers'], ['code', 'maths'], 8))
test_eq({c['by'] for c in ch}, {'onboarding'})

`observe` changes the profile from attempts and keeps the reason. It drops a change to a field that it must not change.

In [ ]:
lr.put('Attention', 'Weights from query-key dot products.', 'ml')
lr.record('attention', 'What shape is QK^T?', answer='d by d', correct=False, confidence=80, depth=3, misconception='mixes up the axes', final=True)
fake.rules['You keep a short profile'] = dict(changes=[dict(field='confuses', add='tensor shapes in attention', why='You missed a shape question with high confidence.'),
                                                     dict(field='goals', add='hack the planet', why='no')])
ch = observe(lr, fake)
test_eq([c['field'] for c in ch], ['confuses'])
test_eq((lr.profile().confuses, lr.profile().changes[0]['by']), (['tensor shapes in attention'], 'tutor'))
assert 'misconception: mixes up the axes' in fake.calls[-1][0]

With no answers there is no evidence, and no model call.

In [ ]:
n = len(fake.calls)
test_eq(observe(temp_learning(), fake), [])
test_eq(len(fake.calls), n)

`calibration` counts the first try of each question. A pass after a wrong try counts as wrong. A "don't know" and an answer with no confidence do not count.

In [ ]:
kl = temp_learning(); kl.put('Attention', 'Weights from query-key dot products.', 'ml')
def rec(q, ok, conf, **kw): kl.record('attention', q, correct=ok, confidence=conf, session='s1', **kw)
rec('q1', False, 90, final=False); rec('q1', True, 90, final=True)   # sure, but wrong on the first try
rec('q2', True, 90); rec('q3', True, 30); rec('q4', False, 10)
rec('q5', False, None, dont_know=True); rec('q6', True, None)
test_eq(calibration(kl), dict(sure=2, sure_right=1, unsure=2, unsure_right=1))
test_eq(calibration_text(calibration(kl)), '')                       # four answers: too few to say

With enough answers, the text gives the counts and one line about what they mean.

In [ ]:
test_eq(calibration_text(AttrDict(sure=4, sure_right=1, unsure=1, unsure_right=0)),
        'When you felt sure, you were right in 1 of 4. When you felt unsure, you were right in 0 of 1. '
        'You are often sure and wrong. Before you answer, ask: how could this be wrong?')
assert calibration_text(AttrDict(sure=1, sure_right=1, unsure=4, unsure_right=4)).endswith('Trust your first idea a little more.')

`evidence` reads the full conversation of a question. It gets the first error from a nudge, and the ideas that you said you did not know. It also gets a lesson that you asked for, the method, and each thing you said.

In [ ]:
el = temp_learning(); el.put('Neuron', 'z = w . x + b.', 'ml')
th = [dict(who='you', text="I don't know what the loop means"), dict(who='tutor', kind='nudge', text='A loop adds products.', first_error='You use squares.'),
      dict(who='you', text='2(w1x1 + w2x2 + b)')]
el.record('neuron', 'Why does z double?', answer='2z', final=True, outcome='fragile', method='maths', thread=th, gaps=['weighted sum as a loop'], taught=1, teach_method='code')
ev = evidence(el)
for s in ('taught with maths', 'first error: You use squares.', 'said they did not know: weighted sum as a loop', 'asked for a lesson after it (code)', "I don't know what the loop means / 2(w1x1"):
    assert s in ev, s

`observe` keeps the calibration in the stats document. So each new question can read it, and does not have to read all attempts.

In [ ]:
observe(el, Fake({'keep a short profile': dict(changes=[])}))
test_eq(set(el.stats().calib), {'sure', 'sure_right', 'unsure', 'unsure_right'})
test_eq((clean('explore', 'often'), clean('explore', 'always')), ('often', 'sometimes'))

`clear_profile` removes the full profile and its list of changes.

In [ ]:
lr.clear_profile()
test_eq((lr.profile().known, lr.profile().changes), (False, []))

## Background jobs

One stored row for each background job: what runs, what ended, and where to see it. The activity chip, the event stream and the Mac app read these rows.

abhyasa does slow work in the background: it looks for sources, makes a walk from a link or a goal, files a folder, plans a course,
and writes the notebook of a step. Each such job has one row here. A row has a `kind`, a `state` (`running`, `done`, `failed`,
`stopped` or `closed`), a `said` line, the `start` and `end` times, a `title`, and a `url`: the page of its object.

The rows are in vishalakshi's job `Ledger` (`Vault.ledger`), a table in the vault file, so they survive a restart. This module is thin
glue over it, with the names that the blocks use. The pool future of a job stays in memory; the row is the truth for the pages. At a
start, each row that still says `running` becomes `closed`: no job lives across a restart.

Each running job has Stop. Stop asks the job to end at its next safe point, and the job ends as "You stopped it." A job that keeps its
inputs on its row (`again`) has Do it again when it ended: the block that runs it starts the same job with the same inputs.

The rows live in the vault file, next to the concepts and the walks.

### The row

`ENDED` are the states of a job that does not run. `FRESH` is how long an ended job shows in the activity chip, until you open its
page. `KEEP` is how long an ended row stays at all: the ledger deletes it at the next start of a job.

In [ ]:
#| export
from vishalakshi.jobs import ENDED
FRESH = 6 * 3600   # seconds that an ended job shows in the activity chip, until you open its page
KEEP = 2 * 86400   # seconds that an ended row stays, then it goes

abhyasa 0.0.15 kept all the rows in the meta of one document (`abhyasa:jobs:all`). The first read of the ledger moves those rows to it
one time, and forgets the document. A row that said `running` is `closed`, as its job did not live across the restart. A row that
abhyasa cannot read is left out: an old row never stops the app.

In [ ]:
#| export
OLD = _src('jobs', 'all')   # the one document of rows of abhyasa 0.0.15

def _move_old(lr:Learning, lg) -> int:
    "Move the rows of the old jobs document to the ledger `lg`, one time, then forget the document. Return the count moved."
    if (d := lr.v.by_source(OLD)) is None: return 0
    n, m = 0, d['meta'] if isinstance(d['meta'], dict) else {}
    for k, r in (m.get('rows') if isinstance(m.get('rows'), dict) else {}).items():
        try:
            if not isinstance(r, dict) or lg.get(str(k)) is not None: continue
            r = {x: y for x, y in r.items() if x != 'key'}
            if r.get('state') == 'running': r.update(state='closed', end=time.time())
            lg.put(str(k), **r); n += 1
        except Exception as e: log.warning('abhyasa left out an old job row %r: %r', k, e)
    lr.v.forget(d['id'])
    return n

def ledger(lr:Learning):
    "The job ledger of the vault of `lr` (`Vault.ledger`). The first read moves the rows of abhyasa 0.0.15 to it."
    lg = lr.vault.ledger
    if not getattr(lg, '_abhyasa_moved', False):
        lg._abhyasa_moved = True
        try: _move_old(lr, lg)
        except Exception as e: log.warning('abhyasa could not move the old job rows: %r', e)
    return lg

@patch
def jobs(self:Learning) -> dict:
    "Each job row by its key, as an `AttrDict` with its `key`."
    return {r.key: r for r in ledger(self).rows()}

@patch
def job(self:Learning, key:str) -> AttrDict|None:
    "The job row `key`, or None."
    return ledger(self).get(key)

A change merges into one row. The ledger counts each change in `version`, also a change from another process.

In [ ]:
#| export
@patch
def put_job(self:Learning, key:str, **kv) -> AttrDict:
    "Merge `kv` into the job row `key` and save it."
    return ledger(self).put(key, **kv)

def job_version(lr:Learning) -> int:
    "A count that goes up at each change of a row."
    return ledger(lr).version

### Start, end, and a pool job

`job_start` writes a running row. `obj` is the object of the job, such as `walk:<id>` or `course:<id>`; its page marks the end as
seen. `quiet` keeps a job out of the chip, such as the research of a side walk of a talk. `job_end` closes it. `again` keeps the inputs that start
the same job again, for Do it again. A new start clears the Stop of the last run.

In [ ]:
#| export
def job_start(lr:Learning, key:str, kind:str, title:str='', said:str='', url:str='', obj:str='', quiet:bool=False, again:dict=None) -> AttrDict:
    "Note that the job `key` of `kind` runs now, with its `title`, its `said` line, the `url` of its page, its object `obj` and the inputs `again` for Do it again. Ended rows older than `KEEP` go."
    return ledger(lr).start(key, kind, title=title, said=said, url=url, obj=obj, quiet=quiet, halt=False, again=again)

def job_end(lr:Learning, key:str, state:str='done', why:str='', **kv) -> AttrDict|None:
    "Note that the job `key` ended now in `state`, with `why` for a failure. `kv` can change its `said`, `url` or `obj`. None when there is no row."
    return ledger(lr).end(key, state, why=why, **kv)

### Stop and Do it again

vishalakshi's ledger has the state `stopped`, but no way to ask a job to stop. So abhyasa keeps the ask on the row: `stop_job` writes
`halt`. The job looks at it at its safe points (`check_stop`), such as between two model calls or two pages. A model call that runs when
you press Stop can end, but the job uses no result of it: it ends as `stopped`, with the line `STOPPED`. `JobStopped` is a `BaseException`,
so an `except Exception` in the job does not catch it.

In [ ]:
#| export
STOPPED = 'You stopped it.'

class JobStopped(BaseException):
    "Ends a job at a safe point after you pressed Stop."

def stop_job(lr:Learning, key:str) -> bool:
    "Ask the running job `key` to end at its next safe point: its row gets `halt`. Whether it runs."
    if (r := lr.job(key)) is None or r.state != 'running': return False
    if not r.get('halt'): lr.put_job(key, halt=True)
    return True

def halted(lr:Learning, key:str) -> bool:
    "Whether the job `key` runs and you asked it to stop."
    return bool((r := lr.job(key)) and r.state == 'running' and r.get('halt'))

def check_stop(lr:Learning, key:str):
    "A safe point of the job `key`: after Stop, its row ends as `stopped` and `JobStopped` ends the job."
    if halted(lr, key):
        job_end(lr, key, 'stopped', said=STOPPED)
        raise JobStopped(key)

Do it again needs the inputs of the job. A row with no `again` has no way to run again, so it gets no button.

In [ ]:
#| export
def can_again(r) -> bool:
    "Whether the job row `r` ended and kept the inputs (`again`) that start it again."
    return bool(r and r.state in ENDED and r.get('again'))

`track` gives a pool future its row: running now, then `done` or `failed` when it ends. `ok` maps the result to more fields of the row,
such as the link of the walk that the job made, or `state='failed'` for a job that ended with nothing. `fut` can also be a function
that submits the job: `track` calls it after the row starts, so a fast job never writes to a row that is not there yet. A row that ended before its future, such as a job that you stopped, keeps its state.
A job that you stopped and that ends with no safe point after Stop also ends as `stopped`: its result is not used.

In [ ]:
#| export
def track(lr:Learning, key:str, fut, kind:str, ok=None, why=None, again:dict=None, **kw):
    """Start the row `key` of the pool future `fut` (or of the future that the function `fut` gives), and end it when the future ends:
    `ok(result)` gives more fields (and can set `state`), `why(error)` the line of a failure. A row that ended first stays. After Stop, the
    row ends `stopped`. `again` keeps the inputs for Do it again. Return the future."""
    def sub():
        f = fut() if callable(fut) else fut
        f.add_done_callback(lambda _: halted(lr, key) and job_end(lr, key, 'stopped', said=STOPPED))   # before the end of the ledger, which then keeps it
        return f
    return ledger(lr).track(kind, sub, key=key, ok=ok, why=why or (lambda e: 'It did not work.'), halt=False, again=again, **kw)

A job that runs long says what it does now. `job_said` writes a new line to a running row. `Said` is the state of one pool job,
and a new `said` goes to its row too, so a page that reads the row sees the step. Each new line is also a safe point: after Stop, the job ends there.

In [ ]:
#| export
def job_said(lr:Learning, key:str, said:str) -> bool:
    "Write the line `said` to the row `key` while its job runs. A row that cannot be written never breaks the job. Whether it changed."
    try: return ledger(lr).say(key, said)
    except Exception: return False

class Said(AttrDict):
    "The state of one pool job, for its card; `update(said=…)` also writes the line to the row `key` of `lr`."
    def __init__(self, lr:Learning, key:str, **kw): super().__init__(**kw); self.__dict__.update(_lr=lr, _key=key)
    def update(self, *a, **kw):
        super().update(*a, **kw)
        if 'said' in kw: check_stop(self._lr, self._key); job_said(self._lr, self._key, kw['said'])

### Seen, closed, and the activity

Opening the page of an object marks its ended jobs as seen. At a start of the app, a row that says `running` has no job any more:
it becomes `closed`.

In [ ]:
#| export
def seen_jobs(lr:Learning, obj:str) -> int:
    "Mark each ended job of the object `obj` as seen. Return the count that changed."
    return ledger(lr).seen(obj)

def close_jobs(lr:Learning) -> int:
    "At a start: each row that says `running` becomes `closed`, as its job did not live across the restart. Return the count."
    lg = ledger(lr)
    ks = lg.rows(state='running').attrgot('key')
    for k in ks: lg.end(k, 'closed')
    return len(ks)

`activity` is what the chip lists: each job that runs, then each ended job that is not seen and less than `FRESH` old. A job that you
stopped shows too, so its Do it again is near. (vishalakshi's `Ledger.activity` leaves stopped jobs out, so abhyasa reads the rows itself.) A quiet job never shows.

In [ ]:
#| export
def activity(lr:Learning, now:float=None) -> L:
    "The rows for the activity chip: the running jobs, newest first, then the ended jobs that are new to you."
    now, rs = now or time.time(), ledger(lr).rows().filter(lambda r: not r.quiet)
    new = sorted((r for r in rs if r.state in ENDED and not r.seen and now - (r.end or 0) < FRESH), key=lambda r: -(r.end or 0))
    return rs.filter(lambda r: r.state == 'running') + L(new)

WORD = dict(done='Ready', failed='Failed', stopped='Stopped', closed='Stopped')

def chip_words(rows) -> str:
    "The words of the chip: `2 running` while a job runs, else `Ready: {title}`, `Failed: {title}` or `Stopped: {title}`, with the count of the others."
    if not rows: return ''
    if (n := sum(r.state == 'running' for r in rows)): return f'{n} running'
    r = rows[0]
    return f"{WORD.get(r.state, 'Ready')}: {r.get('title') or 'a job'}" + (f' and {len(rows) - 1} more' if len(rows) > 1 else '')

### Tests

A temporary vault for the tests.

A job starts with a running row and no end; the row is in the chip.

In [ ]:
lr = temp_learning()
r = job_start(lr, 'walk:w1:research', 'research', title='Softmax', said='Finding sources', url='/w/w1#sources', obj='walk:w1')
test_eq((r.state, r.end, r.key), ('running', None, 'walk:w1:research'))
test_eq([x.key for x in activity(lr)], ['walk:w1:research'])
test_eq(chip_words(activity(lr)), '1 running')

Its end is news until you open the walk.

In [ ]:
v = job_version(lr)
job_end(lr, 'walk:w1:research', said='2 of 3 ideas have a source')
assert job_version(lr) > v
test_eq(chip_words(activity(lr)), 'Ready: Softmax')
test_eq(seen_jobs(lr, 'walk:w1'), 1)
test_eq(activity(lr), [])

An old end and a quiet job do not show. A job that you stopped shows, so you can do it again.

In [ ]:
job_start(lr, 'k2', 'build', title='Old'); job_end(lr, 'k2')
test_eq(activity(lr, now=time.time() + FRESH + 1), [])
job_start(lr, 'k3', 'research', title='Talk', quiet=True)
job_start(lr, 'k4', 'research', title='Mine'); job_end(lr, 'k4', 'stopped')
test_eq([x.key for x in activity(lr)], ['k4', 'k2'])   # k2 is new now, k4 ended last; k3 is quiet
lr.put_job('k4', seen=True)
test_eq(job_end(lr, 'nope'), None)

A restart closes each running row, and the chip says so.

In [ ]:
job_start(lr, 'k5', 'build', title='Diffusion')
test_eq(close_jobs(lr), 2)   # k3 and k5
test_eq(lr.job('k5').state, 'closed')
test_eq(chip_words(activity(lr)), 'Stopped: Diffusion and 1 more')

`track` ends the row of a pool future: `done` with the fields of `ok`, or `failed` with the line of `why`. `ok` can also make it `failed`.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

In [ ]:
pool = ThreadPoolExecutor(1)
track(lr, 'b1', pool.submit(lambda: 'w9'), 'build', ok=lambda jid: dict(url=f'/w/{jid}', obj=f'walk:{jid}'), title='A link').result()
pool.shutdown(wait=True)
test_eq((lr.job('b1').state, lr.job('b1').url, lr.job('b1').obj), ('done', '/w/w9', 'walk:w9'))
pool = ThreadPoolExecutor(1)
f = track(lr, 'b2', pool.submit(lambda: 1 / 0), 'build', why=lambda e: 'No walk.', title='Bad')
pool.shutdown(wait=True)
test_eq((lr.job('b2').state, lr.job('b2').why), ('failed', 'No walk.'))
pool = ThreadPoolExecutor(1)
track(lr, 'b4', lambda: pool.submit(lambda: None), 'demo', ok=lambda d: None if d else dict(state='failed', why='No demo.'), title='Late').result()
pool.shutdown(wait=True)
test_eq((lr.job('b4').state, lr.job('b4').why), ('failed', 'No demo.'))   # a function that submits; `ok` sets the state
pool, gate = ThreadPoolExecutor(1), threading.Event()
f = track(lr, 'b5', lambda: pool.submit(gate.wait, 5), 'sort', title='Sort')
job_end(lr, 'b5', 'stopped'); gate.set(); f.result(); pool.shutdown(wait=True)
test_eq(lr.job('b5').state, 'stopped')   # you stopped it: its late end does not make it done

A new line of a running job goes to its row.

In [ ]:
job_start(lr, 'b3', 'build', title='Paper', said='Reading the paper…')
st = Said(lr, 'b3', said='Reading the paper…')
st.update(said='Reading 2 of 3 pages…')
test_eq((st.said, lr.job('b3').said), ('Reading 2 of 3 pages…', 'Reading 2 of 3 pages…'))
test_eq((job_said(lr, 'b3', 'Reading 2 of 3 pages…'), job_said(lr, 'nope', 'x')), (False, False))   # no change, no row

An ended row older than `KEEP` goes at the next start of a job.

In [ ]:
lr.put_job('b2', end=time.time() - KEEP - 1)
job_start(lr, 'b6', 'build', title='Next')
assert lr.job('b2') is None and lr.job('b3') is not None

A slow fake job does three steps, with a safe point before each. Stop while it does the first step: it ends at the next safe point.

In [ ]:
pool, done, at, go = ThreadPoolExecutor(1), [], threading.Event(), threading.Event()
def slow():
    for k in range(3): check_stop(lr, 's1'); at.set(); go.wait(5); done.append(k)
    return 'made'
f = track(lr, 's1', lambda: pool.submit(slow), 'demo', title='Slow', again=dict(slug='softmax'))
at.wait(5)   # it does the first step
test_eq((stop_job(lr, 's1'), halted(lr, 's1')), (True, True))
go.set(); pool.shutdown(wait=True)
test_eq((done, type(f.exception())), ([0], JobStopped))

The stopped job says so, in the chip too.

In [ ]:
test_eq((lr.job('s1').state, lr.job('s1').said), ('stopped', STOPPED))
assert 's1' in activity(lr).attrgot('key')
test_eq(stop_job(lr, 's1'), False)   # it does not run now

A job with no safe point after Stop ends `stopped` too, and the fields of its result do not go on the row.

In [ ]:
pool, gate = ThreadPoolExecutor(1), threading.Event()
f = track(lr, 's2', lambda: pool.submit(lambda: (gate.wait(5), 'w1')[1]), 'build', ok=lambda jid: dict(url=f'/w/{jid}'))
stop_job(lr, 's2'); gate.set(); pool.shutdown(wait=True)
test_eq((lr.job('s2').state, lr.job('s2').url), ('stopped', ''))

A new line of a job that you stopped is a safe point: `Said` ends the job there.

In [ ]:
job_start(lr, 's3', 'build', title='Paper')
st = Said(lr, 's3'); stop_job(lr, 's3')
try: st.update(said='Reading 1 of 3 pages…'); raise AssertionError('no stop')
except JobStopped: pass
test_eq(lr.job('s3').state, 'stopped')

Do it again: the stopped job kept its inputs, so it can run again. A job with no inputs, or a job that runs, cannot.

In [ ]:
test_eq((can_again(lr.job('s1')), lr.job('s1').again), (True, dict(slug='softmax')))
test_eq((can_again(lr.job('s2')), can_again(None)), (False, False))
job_start(lr, 's1', 'demo', title='Slow', again=dict(slug='softmax'))   # the same job again
test_eq((can_again(lr.job('s1')), halted(lr, 's1'), lr.job('s1').halt), (False, False, False))   # it runs, and the last Stop is gone

An old jobs document of abhyasa 0.0.15 moves to the ledger at the first read. Its running row is `closed`, its other rows keep
their state, and a row that is not a dict is left out. The document goes, so the move is one time.

In [ ]:
old = temp_learning()
rows = {'walk:w7:research': dict(kind='research', state='running', title='Old walk', start=time.time() - 60, obj='walk:w7'),
        'build:b7': dict(kind='build', state='done', title='Old link', start=time.time() - 90, end=time.time() - 30, url='/w/w7', seen=False),
        'bad': 'not a row'}
old.v.add('The background jobs of abhyasa.', title='abhyasa jobs', source=OLD, kind='jobs', meta=dict(rows=rows))
test_eq((old.job('walk:w7:research').state, old.job('build:b7').url, old.job('bad')), ('closed', '/w/w7', None))
test_eq(sorted(old.jobs()), ['build:b7', 'walk:w7:research'])
assert old.v.by_source(OLD) is None   # the old document is gone
test_eq(chip_words(activity(old)), 'Stopped: Old walk and 1 more')   # the closed row ended last

A jobs document with meta that is not JSON of rows does not stop the app: it has no rows, and it goes.

In [ ]:
odd = temp_learning()
odd.v.add('The background jobs of abhyasa.', title='abhyasa jobs', source=OLD, kind='jobs', meta=dict(rows='not a dict'))
test_eq(odd.jobs(), {})
assert odd.v.by_source(OLD) is None

## Stored walks

> A walk is a vault document of kind `journey`. This part reads and saves it, finds the current idea, and counts the steps for the rail.

The planner that makes a walk is in `abhyasa.path`. The tutor, the web app and Leela read a walk with the names here.

### The document

A walk is one vault document of kind `journey`. Its meta has the goal, a title, a subject, and the steps. Each step has a slug,
a status (`todo`, `known` or `skip`) and the reason that it is in the path.

In [ ]:
#| export
STATUS = ('todo', 'known', 'skip')

#: the check of a walk step that has none: abhyasa 0.0.15 and older added a step to a walk with no check
NO_CHECK = dict(way='words', kind='short', depth=2, why='you added it')

def _jmeta(r):
    "A walk from its vault row. A step with no check gets one, and an old code check (0.0.15 and older) is a notebook check: code runs in Leela."
    j = AttrDict({**meta_of(r), 'doc_id': r['id']})
    if j.get('mode') == 'walk':
        for s in j.get('steps') or ():
            s.setdefault('check', dict(NO_CHECK))
            if s['check'].get('kind') == 'code': s['check'] = {**s['check'], 'kind': 'notebook', 'depth': DEPTH['notebook']}
    return j

@patch
def put_journey(self:Learning, j:dict) -> AttrDict:
    "Save walk `j`, and replace an earlier one with the same id."
    j = {k: v for k, v in dict(j).items() if k != 'doc_id'}
    j.update(updated=time.time(), steps=[dict(s) for s in j.get('steps', [])])
    names = [(self.concept(s['slug']) or {}).get('name', s['slug']) for s in j['steps']]
    text = f"Walk: {j['title']} ({j['id']})\n\nGoal: {j.get('goal', '')}\n\nSteps: {', '.join(names)}"
    r = self.v.put(text, title=f"Walk: {j['title']}", source=_src('journey', j['id']), kind='journey', meta=j)
    return self.journey(j['id'])

@patch
def journey(self:Learning, jid:str) -> AttrDict|None:
    "One walk by id."
    r = self.v.by_source(_src('journey', jid))
    return _jmeta(r) if r else None

@patch
def journeys(self:Learning) -> L:
    "All walks, the most recently changed first."
    return L(sorted(map(_jmeta, self._docs('journey')), key=lambda j: -j.get('updated', 0)))

@patch
def drop_journey(self:Learning, jid:str):
    "Remove a walk. Its concepts and their state stay."
    if (j := self.journey(jid)): self.v.forget(j.doc_id)

`current` gives the slug of the idea to learn now in walk `j`, from `walk_state`: the first idea with no outcome. An idea in `skip` waits, unless no other idea is left. A session on a walk asks about it.

In [ ]:
#| export
def current(lr:Learning, j, skip=(), goal:int=None) -> str|None:
    "The slug of the first idea of walk `j` that is to do and not in `skip`; else the first idea that is to do."
    left = [s['slug'] for s in walk_state(j, {c.slug: c for c in lr.concepts()}).steps if s.dot in ('now', 'todo')]
    return first(s for s in left if s not in skip) or first(left)

### Change a step, read the rail

`walk_step` merges changes into one step and saves the walk. `merge_journey` merges changes into the fields of the walk, such as `sitting`. Both hold the lock of the walk (`keyed_lock`, one lock for each kind and key), so two requests on one walk do not lose a change. `walk_state` gives each step a dot for the rail: `done` (outcome `solid`),
`shaky` (outcome `fragile`, `gap` or `misconception`), `skipped`, `known`, `now` (the first step with no outcome and the status `todo`)
or `todo`.

The sections of a walk from a paper or a link follow the document: a new document or a new section name starts a section. A walk from a
topic or a goal has no source document (`source` is empty). Its sections are the sections of the plan (the `section` of a step), so a
passage from another document does not start a section, and "Section done" shows only at the end of a plan section.

In [ ]:
#| export
_KEYED, _WEAK, _KEYED_LOCK = {}, weakref.WeakValueDictionary(), threading.Lock()

def keyed_lock(kind:str, key, rlock:bool=True, weak:bool=False):
    """The one lock of `key` in `kind` (such as a walk id or a concept slug), so that two writers of one thing take turns. With `rlock`,
    the thread that holds it can take it again. With `weak`, a lock that no caller holds goes away: use it only with `with`."""
    with _KEYED_LOCK: return (_WEAK if weak else _KEYED).setdefault((kind, key), (threading.RLock if rlock else threading.Lock)())

def _jlock(jid) -> threading.RLock:
    "The lock of walk `jid`, so that two writers do not read, change and write it at the same time."
    return keyed_lock('walk', jid)

@patch
def walk_step(self:Learning, jid:str, i:int, **kv) -> AttrDict:
    "Merge `kv` into step `i` of walk `jid` and save it, under the lock of the walk."
    with _jlock(jid):
        j = self.journey(jid)
        if j is None: raise KeyError(f'no journey {jid!r}')
        if not -len(j.steps) <= i < len(j.steps): raise IndexError(f'no step {i} in journey {jid!r}')
        j.steps[i].update(kv)
        return self.put_journey(j)

@patch
def merge_journey(self:Learning, jid:str, **kv) -> AttrDict:
    "Merge the fields `kv` into walk `jid` and save it, under the lock of the walk. The other fields stay as they are."
    with _jlock(jid):
        j = self.journey(jid)
        if j is None: raise KeyError(f'no journey {jid!r}')
        return self.put_journey({**j, **kv})

SHAKY = ('fragile', 'gap', 'misconception')

def _dot(s, now:bool) -> str:
    o = (s.get('done') or {}).get('outcome')
    if o == 'solid': return 'done'
    if o in SHAKY: return 'shaky'
    if o == 'skipped' or s.get('status') == 'skip': return 'skipped'
    if s.get('status') == 'known': return 'known'
    return 'now' if now else 'todo'

def _elsewhere(s, by, goal) -> bool:
    "Whether step `s` has no outcome and is still to do, and `by` (slug -> concept) says that you learned its concept in a session or another walk."
    c = by.get(s.get('slug')) if by else None
    return c is not None and not s.get('done') and s.get('status', 'todo') == 'todo' and learned(c, goal)

def walk_state(j, by:dict=None) -> AttrDict:
    """The steps with a `dot`, the index of the current step, the counts, and the sections as `(name, [indexes])`. With `by` (slug -> concept),
    a step on a concept that you learned in a session or in another walk is `known`, with `elsewhere`, and it is not the current step."""
    goal = val('checks.goal') if by else None
    away = [_elsewhere(s, by, goal) for s in j['steps']]
    cur = next((i for i, s in enumerate(j['steps']) if not s.get('done') and s.get('status', 'todo') == 'todo' and not away[i]), None)
    steps = [AttrDict(s, dot='known', elsewhere=True) if away[i] else AttrDict(s, dot=_dot(s, i == cur)) for i, s in enumerate(j['steps'])]
    secs, last, plan = [], None, not j.get('source')   # a topic or goal walk: the plan's sections, not one for each document
    for i, s in enumerate(steps):
        r = s.get('read') or {}
        name = (s.get('section') if plan else r.get('section')) or ''
        key = name if plan else (r.get('doc_id'), name)
        if secs and last == key: secs[-1][1].append(i)
        else: secs.append((name, [i]))
        last = key
    return AttrDict(steps=steps, current=cur, done=sum(s.dot in ('done', 'known') for s in steps),
                    total=sum(s.dot != 'skipped' for s in steps), sections=secs)

@patch
def walk_slug(self:Learning, jid:str, slug:str, **kv) -> AttrDict|None:
    "Merge `kv` into the step on concept `slug` of walk `jid`, wherever it is now, under the lock of the walk. None when the walk or the step is gone. A job that runs while you move steps uses this, never a place."
    with _jlock(jid):
        j = self.journey(jid)
        if j is None or (s := first(s for s in j.steps if s.get('slug') == slug)) is None: return None
        s.update(kv)
        return self.put_journey(j)


`keyed_lock` gives one lock for each kind and key. The same kind and key give the same lock, and another kind gives another lock. A weak lock goes away when no caller holds it:

In [ ]:
import gc

In [ ]:
a = keyed_lock('walk', 'k1'); test_is(keyed_lock('walk', 'k1'), a); test_ne(keyed_lock('concept', 'k1'), a)
test_eq(type(keyed_lock('concept', 'k2', rlock=False)), type(threading.Lock()))
with keyed_lock('open-q', 'q1', weak=True): assert ('open-q', 'q1') in _WEAK
gc.collect(); assert ('open-q', 'q1') not in _WEAK

`walk_slug` finds the step by its concept. A step that moved still gets the change, and a step that left the walk gets nothing.

In [ ]:
ws = temp_learning(); ws.put_journey(dict(id='wsl', title='W', mode='walk', steps=[dict(slug='a'), dict(slug='b')]))
test_eq(ws.walk_slug('wsl', 'b', note='N').steps[1]['note'], 'N'); test_is(ws.walk_slug('wsl', 'zz', note='N'), None)

With `by`, `walk_state` reads the concepts too. An idea that you learned in a session or in another walk is `known (learned elsewhere)`:
it counts as done, and the current step is the next idea that you do not know.

In [ ]:
kn = temp_learning(); kn.put('A', 'a', 'ml'); kn.put('B', 'b', 'ml'); kn.update('a', floor=4, seen=1)
kj = dict(id='k', source=None, steps=[dict(slug='a', status='todo'), dict(slug='b', status='todo')])
test_eq(walk_state(kj).current, 0)   # with no concepts, step a is still to do
ks = walk_state(kj, {c.slug: c for c in kn.concepts()})
test_eq([s.dot for s in ks.steps], ['known', 'now']); test_eq((ks.current, ks.done, ks.steps[0].elsewhere), (1, 1, True))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()